# 🧠 Agent Evaluation: From Zero to Production
### A Complete LangGraph Curriculum — Basic Agents → Complex Workflows → Multi-Agent → MCP → Evaluation

---

**Stack:** LangGraph · LangChain · DeepEval · RAGAS · Pure Python Evaluators  
**Philosophy:** Every concept is *shown*, *run*, and *evaluated*. No black boxes.

---

## 📋 Table of Contents

| # | Section | Topics |
|---|---------|--------|
| 1 | **Environment & Architecture Primer** | Install, imports, LangGraph mental model |
| 2 | **Basic Agents** | ReAct agent, tool-calling, state machines |
| 3 | **Complex Single Agents** | Planning, memory, reflection, self-critique |
| 4 | **Multi-Agent Systems** | Supervisor, subgraph, handoff patterns |
| 5 | **MCP & Advanced Tool Calling** | MCP protocol, dynamic tools, parallel calls |
| 6 | **Agentic Workflows** | Sequential, parallel, map-reduce, conditional |
| 7 | **Agent Evaluation — Foundations** | Metrics taxonomy, trace logging, test design |
| 8 | **Evaluation Frameworks** | DeepEval, RAGAS, custom evaluators |
| 9 | **Advanced Evaluation** | Multi-turn, multi-agent, latency, safety |
| 10 | **Production Evaluation Layer** | CI/CD eval, regression suites, dashboards |

---
> 💡 **How to use this notebook:** Run cells sequentially. All LLM calls use mock/stub LLMs by default — replace with your real API key where indicated. Every agent built here has a corresponding evaluation section.


In [1]:
# ─── 📦 SECTION 1: ENVIRONMENT SETUP ────────────────────────────────────────
# Run this once. All packages are open-source.

import subprocess, sys

packages = [
    "langgraph>=1.2",
    "langchain>=1.3",
    "langchain-core",
    "langchain-community",
    "langchain-openai",
    "deepeval>=4.0",
    "ragas>=0.4",
    "pandas",
    "numpy",
    "matplotlib",
    "rich",
    "tabulate",
]

def install(pkgs):
    for p in pkgs:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", p, "-q", "--break-system-packages"],
            capture_output=True
        )
    print("✅ All packages ready")

install(packages)


✅ All packages ready


In [2]:
# ─── 🔧 CORE IMPORTS ─────────────────────────────────────────────────────────
import os, json, time, uuid, asyncio, random, re, copy
from typing import (
    TypedDict, Annotated, Sequence, Literal, Optional, List, Dict, Any, Callable
)
from dataclasses import dataclass, field
from datetime import datetime
import operator

# LangGraph
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode, tools_condition

# LangChain core
from langchain_core.messages import (
    BaseMessage, HumanMessage, AIMessage, SystemMessage, ToolMessage
)
from langchain_core.tools import tool
from langchain_core.runnables import RunnableConfig

# Visualization
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.progress import Progress, SpinnerColumn, TextColumn
from rich import print as rprint

console = Console()
rprint("[bold green]✅ Core imports successful![/bold green]")
rprint(f"  LangGraph, LangChain, DeepEval, RAGAS, Matplotlib, Rich — all loaded")


C:\Users\pc\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Core imports successful!

LangGraph, LangChain, DeepEval, RAGAS, Matplotlib, Rich — all loaded

## 1.1 LangGraph Mental Model

LangGraph models agents as **cyclic directed graphs** over a **shared state object**.

```
┌─────────────────────────────────────────────────────────┐
│                    LANGGRAPH GRAPH                       │
│                                                          │
│   START ──► [Node A] ──► [Node B] ──► [Node C] ──► END  │
│                  ▲           │                           │
│                  └───────────┘  (cycle = the agent loop) │
│                                                          │
│   STATE = shared TypedDict passed between every node     │
│   EDGES = routing logic (conditional or direct)          │
│   NODES = Python functions that read/write state         │
└─────────────────────────────────────────────────────────┘
```

### Key Primitives

| Primitive | What it is | When to use |
|-----------|-----------|-------------|
| `StateGraph` | Graph builder | Always |
| `TypedDict` | State schema | Always |
| `add_messages` | Auto-appends messages | Conversational agents |
| `MemorySaver` | In-memory checkpointer | Persistence / multi-turn |
| `ToolNode` | Pre-built tool executor | Tool-calling agents |
| `START / END` | Graph boundary nodes | Always |
| `Command` | Return routing+state together | Complex conditional routing |

> **Golden rule:** Nodes *transform* state. Edges *route* between nodes. State is *immutable per step* — nodes return a new partial state.


In [3]:
# ─── 🤖 MOCK LLM INFRASTRUCTURE ──────────────────────────────────────────────
# We build deterministic mock LLMs so the entire notebook runs WITHOUT an API key.
# Every mock follows the real LangChain BaseMessage interface.
# 🔑 To use a real LLM: replace MockLLM() with ChatOpenAI(model="gpt-4o", api_key="...")

class MockResponse:
    """Simulates an AIMessage with optional tool_calls"""
    def __init__(self, content: str, tool_calls: list = None, usage: dict = None):
        self.content = content
        self.tool_calls = tool_calls or []
        self.usage_metadata = usage or {"input_tokens": 50, "output_tokens": 30}
        self.id = f"mock-{uuid.uuid4().hex[:8]}"

    def __repr__(self):
        return f"MockResponse(content={self.content!r}, tool_calls={len(self.tool_calls)})"


class MockLLM:
    """
    Deterministic mock LLM that returns scripted responses.
    Tracks all calls for evaluation purposes.
    
    Usage:
        llm = MockLLM(responses=["Hello!", "The answer is 42."])
        llm.invoke([HumanMessage(content="Hi")])  → MockResponse("Hello!")
    """
    def __init__(self, responses: list = None, tool_calls_sequence: list = None, 
                 latency_ms: float = 50):
        self._responses = responses or ["I am a mock LLM response."]
        self._tool_calls_seq = tool_calls_sequence or []
        self._latency_ms = latency_ms
        self._call_count = 0
        self.call_log = []  # ← evaluation hook
        self.name = "mock-llm"

    def invoke(self, messages: list, config: dict = None) -> MockResponse:
        start = time.time()
        time.sleep(self._latency_ms / 1000)
        
        idx = self._call_count % len(self._responses)
        content = self._responses[idx]
        
        tool_calls = []
        if self._tool_calls_seq and self._call_count < len(self._tool_calls_seq):
            tool_calls = self._tool_calls_seq[self._call_count] or []
        
        response = MockResponse(content=content, tool_calls=tool_calls)
        
        # Log for evaluation
        self.call_log.append({
            "call_id": self._call_count,
            "input": [m.content if hasattr(m, "content") else str(m) for m in messages],
            "output": content,
            "tool_calls": tool_calls,
            "latency_ms": (time.time() - start) * 1000,
            "timestamp": datetime.now().isoformat(),
        })
        
        self._call_count += 1
        return response

    def bind_tools(self, tools):
        """Return self — mock doesn't need real binding"""
        return self

    def with_structured_output(self, schema):
        """Return a structured output mock"""
        return MockStructuredLLM(schema, self._responses, self._latency_ms)


class MockStructuredLLM(MockLLM):
    def __init__(self, schema, responses, latency_ms):
        super().__init__(responses=responses, latency_ms=latency_ms)
        self._schema = schema
    
    def invoke(self, messages, config=None):
        raw = super().invoke(messages, config)
        try:
            parsed = json.loads(raw.content)
        except:
            parsed = {"result": raw.content}
        return parsed


# ── Quick sanity check ──────────────────────────────────────────────────────
llm_demo = MockLLM(responses=["Hello, I'm your agent!", "I'll help you with that."])
resp1 = llm_demo.invoke([HumanMessage(content="Hi")])
resp2 = llm_demo.invoke([HumanMessage(content="Can you help?")])

table = Table(title="Mock LLM Demo", show_header=True, header_style="bold cyan")
table.add_column("Call #"); table.add_column("Input"); table.add_column("Output")
for log in llm_demo.call_log:
    table.add_row(str(log["call_id"]), log["input"][0], log["output"])
console.print(table)
rprint(f"[green]✅ MockLLM works. Total calls logged: {len(llm_demo.call_log)}[/green]")


                    Mock LLM Demo                    
┏━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Call # ┃ Input         ┃ Output                   ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 0      │ Hi            │ Hello, I'm your agent!   │
│ 1      │ Can you help? │ I'll help you with that. │
└────────┴───────────────┴──────────────────────────┘

✅ MockLLM works. Total calls logged: 2

---
## 🤖 Section 2: Basic Agents

We build agents from the ground up:

1. **Hello World Agent** — simplest possible LangGraph graph
2. **ReAct Agent** — Reasoning + Acting loop (the canonical agent pattern)
3. **Tool-Calling Agent** — agents with structured tool use
4. **Stateful Conversational Agent** — memory across turns

Each agent is immediately followed by basic evaluation.


In [4]:
# ─── 2.1 HELLO WORLD AGENT ────────────────────────────────────────────────────
# Simplest LangGraph graph: START → chatbot → END
# State has one field: messages (auto-appended via add_messages reducer)

class BasicState(TypedDict):
    messages: Annotated[list, add_messages]  # ← add_messages is the reducer


# Every node is a plain Python function: (state) → partial state update
def chatbot_node(state: BasicState) -> dict:
    llm = MockLLM(responses=[
        "Hello! I'm a basic LangGraph agent. How can I help you today?",
        "That's an interesting question. Let me think about it...",
        "Great talking with you! Goodbye.",
    ])
    last_msg = state["messages"][-1]
    response = llm.invoke(state["messages"])
    return {"messages": [AIMessage(content=response.content)]}


# Build the graph
def build_hello_world_agent():
    builder = StateGraph(BasicState)
    builder.add_node("chatbot", chatbot_node)
    builder.add_edge(START, "chatbot")   # entry point
    builder.add_edge("chatbot", END)     # exit point
    return builder.compile()

# Compile
hello_agent = build_hello_world_agent()

# Run
initial_state = {"messages": [HumanMessage(content="Hello, agent!")]}
result = hello_agent.invoke(initial_state)

console.print(Panel(
    f"[bold]Input:[/bold]  {initial_state['messages'][0].content}\n"
    f"[bold]Output:[/bold] {result['messages'][-1].content}\n\n"
    f"[dim]Message history length: {len(result['messages'])}[/dim]",
    title="[cyan]2.1 Hello World Agent[/cyan]",
    border_style="cyan"
))


╭───────────────────────────────────────────── 2.1 Hello World Agent ─────────────────────────────────────────────╮
│ Input:  Hello, agent!                                                                                           │
│ Output: Hello! I'm a basic LangGraph agent. How can I help you today?                                           │
│                                                                                                                 │
│ Message history length: 2                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [5]:
# ─── 2.2 REACT AGENT (Reasoning + Acting) ─────────────────────────────────────
# ReAct = interleave Thought → Action → Observation loops until done.
# Pattern: agent node ↔ tools node, with conditional exit.

# ── Define tools ─────────────────────────────────────────────────────────────
@tool
def calculator(expression: str) -> str:
    """Evaluate a simple mathematical expression. Input: a math expression string."""
    try:
        # Safe eval for basic math only
        allowed = set("0123456789+-*/()., ")
        if not all(c in allowed for c in expression):
            return "Error: only basic math operators allowed"
        result = eval(expression, {"__builtins__": {}}, {})
        return f"{result}"
    except Exception as e:
        return f"Error: {e}"


@tool
def web_search_mock(query: str) -> str:
    """Mock web search. Returns a canned result for demo purposes."""
    mock_db = {
        "python": "Python is a high-level, interpreted programming language created by Guido van Rossum.",
        "langgraph": "LangGraph is a library for building stateful, multi-actor applications with LLMs.",
        "agent": "An AI agent is a system that perceives its environment and takes actions autonomously.",
        "evaluation": "Agent evaluation measures quality, accuracy, latency, and safety of AI agents.",
    }
    for keyword, result in mock_db.items():
        if keyword.lower() in query.lower():
            return result
    return f"Search results for '{query}': No specific results found. General knowledge applies."


@tool
def get_weather(city: str) -> str:
    """Get mock weather for a city."""
    mock_weather = {
        "jaipur": "Jaipur: 38°C, Sunny, Humidity 25%",
        "mumbai": "Mumbai: 32°C, Partly Cloudy, Humidity 75%",
        "delhi": "Delhi: 41°C, Clear, Humidity 20%",
    }
    return mock_weather.get(city.lower(), f"{city}: 25°C, Clear skies")


TOOLS = [calculator, web_search_mock, get_weather]
TOOL_MAP = {t.name: t for t in TOOLS}

rprint(f"[green]Tools registered:[/green] {[t.name for t in TOOLS]}")

# ── ReAct State ───────────────────────────────────────────────────────────────
class ReActState(TypedDict):
    messages: Annotated[list, add_messages]
    thought: str          # ← ReAct reasoning trace
    action_count: int     # ← guard against infinite loops
    max_steps: int


# ── Agent node: generates thought + action ────────────────────────────────────
def react_agent_node(state: ReActState) -> dict:
    """
    ReAct reasoning step.
    In production: LLM with tool_calls. Here: scripted mock for determinism.
    """
    last_msg = state["messages"][-1].content
    step = state.get("action_count", 0)
    
    # Scripted ReAct for demo — replace with: llm.bind_tools(TOOLS).invoke(messages)
    if step == 0:
        # First step: decide to use calculator
        thought = "I need to compute this. I'll use the calculator tool."
        tool_call = {
            "id": f"call_{uuid.uuid4().hex[:8]}",
            "name": "calculator",
            "args": {"expression": "2 * 3.14159 * 7"},  # circumference of circle r=7
        }
        ai_msg = AIMessage(
            content=f"Thought: {thought}",
            tool_calls=[tool_call]
        )
        return {
            "messages": [ai_msg],
            "thought": thought,
            "action_count": step + 1,
        }
    elif step == 1:
        # Second step: use web search
        thought = "Let me also search for context on this topic."
        tool_call = {
            "id": f"call_{uuid.uuid4().hex[:8]}",
            "name": "web_search_mock",
            "args": {"query": "agent evaluation best practices"},
        }
        ai_msg = AIMessage(
            content=f"Thought: {thought}",
            tool_calls=[tool_call]
        )
        return {
            "messages": [ai_msg],
            "thought": thought,
            "action_count": step + 1,
        }
    else:
        # Final step: synthesize answer
        tool_results = [
            m.content for m in state["messages"] 
            if isinstance(m, ToolMessage)
        ]
        synthesis = f"Based on my calculations and research: {'; '.join(tool_results[:2])}. Task complete."
        return {
            "messages": [AIMessage(content=synthesis)],
            "thought": "Synthesizing results",
            "action_count": step + 1,
        }


# ── Tool execution node ───────────────────────────────────────────────────────
def tool_execution_node(state: ReActState) -> dict:
    """Execute all tool calls from the last AI message."""
    last_msg = state["messages"][-1]
    tool_messages = []
    
    for tc in last_msg.tool_calls:
        tool_fn = TOOL_MAP.get(tc["name"])
        if tool_fn:
            result = tool_fn.invoke(tc["args"])
        else:
            result = f"Tool '{tc['name']}' not found"
        
        tool_messages.append(ToolMessage(
            content=str(result),
            tool_call_id=tc["id"],
            name=tc["name"],
        ))
    
    return {"messages": tool_messages}


# ── Routing logic ─────────────────────────────────────────────────────────────
def should_continue(state: ReActState) -> Literal["tools", "end"]:
    last_msg = state["messages"][-1]
    max_steps = state.get("max_steps", 5)
    
    if state.get("action_count", 0) >= max_steps:
        return "end"
    if isinstance(last_msg, AIMessage) and last_msg.tool_calls:
        return "tools"
    return "end"


# ── Build ReAct Graph ─────────────────────────────────────────────────────────
def build_react_agent():
    builder = StateGraph(ReActState)
    
    builder.add_node("agent", react_agent_node)
    builder.add_node("tools", tool_execution_node)
    
    builder.add_edge(START, "agent")
    builder.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
    builder.add_edge("tools", "agent")  # ← the ReAct loop
    
    return builder.compile()


react_agent = build_react_agent()

# ── Run ───────────────────────────────────────────────────────────────────────
react_result = react_agent.invoke({
    "messages": [HumanMessage(content="What is the circumference of a circle with radius 7? Also search for agent evaluation.")],
    "thought": "",
    "action_count": 0,
    "max_steps": 5,
})

rprint(Panel("[bold cyan]2.2 ReAct Agent Execution Trace[/bold cyan]"))
for i, msg in enumerate(react_result["messages"]):
    role = type(msg).__name__
    content = msg.content[:120] + ("..." if len(msg.content) > 120 else "")
    tool_info = ""
    if isinstance(msg, AIMessage) and msg.tool_calls:
        tool_info = f"  [yellow]→ calls: {[tc['name'] for tc in msg.tool_calls]}[/yellow]"
    rprint(f"  [bold]{i}. {role}:[/bold] {content}{tool_info}")

rprint(f"\n[green]✅ ReAct completed in {react_result['action_count']} steps[/green]")


Tools registered: ['calculator', 'web_search_mock', 'get_weather']

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 2.2 ReAct Agent Execution Trace                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

0. HumanMessage: What is the circumference of a circle with radius 7? Also search for agent evaluation.

1. AIMessage: Thought: I need to compute this. I'll use the calculator tool.  → calls: ['calculator']

2. ToolMessage: 43.98226

3. AIMessage: Thought: Let me also search for context on this topic.  → calls: ['web_search_mock']

4. ToolMessage: An AI agent is a system that perceives its environment and takes actions autonomously.

5. AIMessage: Based on my calculations and research: 43.98226; An AI agent is a system that perceives its 
environment and takes action...

✅ ReAct completed in 3 steps

In [6]:
# ─── 2.3 STATEFUL CONVERSATIONAL AGENT (Multi-turn Memory) ───────────────────
# Uses MemorySaver as a checkpointer → agent remembers across turns
# Key: thread_id in config → separate memory per conversation

class ConvState(TypedDict):
    messages: Annotated[list, add_messages]
    user_name: str
    turn_count: int


scripted_conv_responses = [
    "Nice to meet you! I'll remember your name throughout our conversation.",
    "Yes, I remember — you told me your name at the start. How can I help further?",
    "That's a great question about agent evaluation! It covers accuracy, latency, safety, and more.",
    "Thanks for the conversation! I'm glad I could help you understand agent evaluation.",
]


def conv_node(state: ConvState) -> dict:
    llm = MockLLM(responses=scripted_conv_responses)
    response = llm.invoke(state["messages"])
    return {
        "messages": [AIMessage(content=response.content)],
        "turn_count": state.get("turn_count", 0) + 1,
    }


def build_conv_agent():
    builder = StateGraph(ConvState)
    builder.add_node("chat", conv_node)
    builder.add_edge(START, "chat")
    builder.add_edge("chat", END)
    
    memory = MemorySaver()  # ← persistence checkpointer
    return builder.compile(checkpointer=memory)


conv_agent = build_conv_agent()

# ── Simulate multi-turn conversation ──────────────────────────────────────────
thread_config = {"configurable": {"thread_id": "user-harsh-001"}}

turns = [
    "Hi! My name is Harsh.",
    "Do you remember my name?",
    "What is agent evaluation?",
    "Goodbye!",
]

rprint(Panel("[bold]2.3 Multi-Turn Conversation with Memory[/bold]", border_style="green"))
for turn_input in turns:
    result = conv_agent.invoke(
        {"messages": [HumanMessage(content=turn_input)], "user_name": "", "turn_count": 0},
        config=thread_config,
    )
    human_msg = turn_input
    agent_msg = result["messages"][-1].content
    turn = result.get("turn_count", "?")
    
    rprint(f"  [bold blue]Turn {turn} Human:[/bold blue] {human_msg}")
    rprint(f"  [bold green]Agent:[/bold green] {agent_msg}")
    rprint()

rprint(f"[green]✅ Memory across {len(turns)} turns demonstrated[/green]")
rprint("[dim]thread_id='user-harsh-001' keeps state isolated per user[/dim]")


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 2.3 Multi-Turn Conversation with Memory                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Turn 1 Human: Hi! My name is Harsh.

Agent: Nice to meet you! I'll remember your name throughout our conversation.

Turn 1 Human: Do you remember my name?

Agent: Nice to meet you! I'll remember your name throughout our conversation.

...

Agent: Nice to meet you! I'll remember your name throughout our conversation.

Turn 1 Human: Goodbye!

Agent: Nice to meet you! I'll remember your name throughout our conversation.

✅ Memory across 4 turns demonstrated

thread_id='user-harsh-001' keeps state isolated per user

In [7]:
# ─── 2.4 BASIC AGENT EVALUATION ──────────────────────────────────────────────
# Before reaching DeepEval/RAGAS, understand the FUNDAMENTALS of what we measure.
# This section implements evaluators from scratch — pure Python, no frameworks.

# ════════════════════════════════════════════════════════════════
# CONCEPT: What do we evaluate in agents?
#
#  ┌─────────────────────────────────────────────────────┐
#  │  AGENT QUALITY DIMENSIONS                           │
#  │                                                     │
#  │  1. CORRECTNESS   — Did the agent answer correctly? │
#  │  2. TOOL USE      — Did it call the right tools?    │
#  │  3. EFFICIENCY    — Minimal steps, no redundancy?   │
#  │  4. SAFETY        — No harmful outputs?             │
#  │  5. LATENCY       — Response time acceptable?       │
#  │  6. FAITHFULNESS  — Grounded in context/tools?      │
#  └─────────────────────────────────────────────────────┘
# ════════════════════════════════════════════════════════════════

@dataclass
class AgentTestCase:
    """A single agent evaluation test case"""
    test_id: str
    input: str
    expected_output: str        # reference answer
    expected_tools: List[str]   # tools that should be called
    actual_output: str = ""
    actual_tools: List[str] = field(default_factory=list)
    latency_ms: float = 0.0
    metadata: Dict = field(default_factory=dict)


@dataclass  
class EvalResult:
    """Result of a single metric evaluation"""
    metric: str
    score: float
    passed: bool
    reason: str
    details: Dict = field(default_factory=dict)


class BasicAgentEvaluator:
    """
    Pure Python agent evaluator — no external dependencies.
    Implements: exact_match, tool_accuracy, step_efficiency, keyword_coverage
    """
    
    def __init__(self, pass_threshold: float = 0.7):
        self.threshold = pass_threshold
        self.results: List[EvalResult] = []

    def exact_match(self, tc: AgentTestCase) -> EvalResult:
        """Binary: does output exactly match expected?"""
        score = 1.0 if tc.actual_output.strip() == tc.expected_output.strip() else 0.0
        return EvalResult(
            metric="exact_match", score=score,
            passed=score >= self.threshold,
            reason=f"{'Match' if score else 'No match'}: got '{tc.actual_output[:50]}...'"
        )

    def keyword_coverage(self, tc: AgentTestCase) -> EvalResult:
        """Fraction of expected keywords present in actual output."""
        keywords = tc.expected_output.lower().split()
        # Filter to meaningful words (>3 chars)
        keywords = [w for w in keywords if len(w) > 3]
        if not keywords:
            return EvalResult("keyword_coverage", 1.0, True, "No keywords to check")
        
        actual_lower = tc.actual_output.lower()
        hits = sum(1 for kw in keywords if kw in actual_lower)
        score = hits / len(keywords)
        
        return EvalResult(
            metric="keyword_coverage", score=round(score, 3),
            passed=score >= self.threshold,
            reason=f"{hits}/{len(keywords)} keywords matched",
            details={"keywords": keywords, "hits": hits}
        )

    def tool_accuracy(self, tc: AgentTestCase) -> EvalResult:
        """Did the agent call exactly the right tools?"""
        expected_set = set(tc.expected_tools)
        actual_set = set(tc.actual_tools)
        
        if not expected_set and not actual_set:
            return EvalResult("tool_accuracy", 1.0, True, "No tools expected or called")
        
        precision = len(expected_set & actual_set) / len(actual_set) if actual_set else 0.0
        recall = len(expected_set & actual_set) / len(expected_set) if expected_set else 1.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        
        missing = expected_set - actual_set
        extra = actual_set - expected_set
        
        return EvalResult(
            metric="tool_accuracy", score=round(f1, 3),
            passed=f1 >= self.threshold,
            reason=f"F1={f1:.2f} | missing={missing} | extra={extra}",
            details={"precision": precision, "recall": recall, "f1": f1, "missing": list(missing), "extra": list(extra)}
        )

    def step_efficiency(self, tc: AgentTestCase, actual_steps: int, optimal_steps: int) -> EvalResult:
        """How efficient was the agent? 1.0 = optimal, 0.0 = way too many steps."""
        if actual_steps <= 0 or optimal_steps <= 0:
            return EvalResult("step_efficiency", 0.0, False, "Invalid step counts")
        
        ratio = optimal_steps / actual_steps
        score = min(1.0, ratio)  # never penalize for being under optimal
        
        return EvalResult(
            metric="step_efficiency", score=round(score, 3),
            passed=score >= self.threshold,
            reason=f"Used {actual_steps} steps (optimal: {optimal_steps}). Ratio: {ratio:.2f}"
        )

    def latency_check(self, tc: AgentTestCase, sla_ms: float = 5000) -> EvalResult:
        """Did the agent respond within SLA?"""
        score = 1.0 if tc.latency_ms <= sla_ms else max(0.0, 1.0 - (tc.latency_ms - sla_ms) / sla_ms)
        return EvalResult(
            metric="latency_check", score=round(score, 3),
            passed=tc.latency_ms <= sla_ms,
            reason=f"Latency: {tc.latency_ms:.0f}ms (SLA: {sla_ms:.0f}ms)"
        )

    def evaluate(self, tc: AgentTestCase, actual_steps: int = 3, optimal_steps: int = 2) -> List[EvalResult]:
        """Run all metrics against a test case"""
        return [
            self.keyword_coverage(tc),
            self.tool_accuracy(tc),
            self.step_efficiency(tc, actual_steps, optimal_steps),
            self.latency_check(tc),
        ]


# ── Build test suite for the ReAct agent ─────────────────────────────────────
test_cases = [
    AgentTestCase(
        test_id="TC-001",
        input="What is the circumference of a circle with radius 7?",
        expected_output="43.98 approximately. Circumference = 2 * pi * radius",
        expected_tools=["calculator"],
        actual_output="Based on my calculations: 43.98229715. Task complete.",
        actual_tools=["calculator", "web_search_mock"],
        latency_ms=280,
    ),
    AgentTestCase(
        test_id="TC-002",
        input="Search for information about LangGraph",
        expected_output="LangGraph is a library for building stateful multi-actor LLM applications",
        expected_tools=["web_search_mock"],
        actual_output="LangGraph is a library for building stateful, multi-actor applications with LLMs.",
        actual_tools=["web_search_mock"],
        latency_ms=150,
    ),
    AgentTestCase(
        test_id="TC-003",
        input="What is the weather in Jaipur?",
        expected_output="Jaipur weather: hot and sunny",
        expected_tools=["get_weather"],
        actual_output="Task complete.",  # ← deliberately bad output for demo
        actual_tools=[],  # ← no tools called — bug!
        latency_ms=6500,  # ← SLA violation
    ),
]

# ── Run evaluation ────────────────────────────────────────────────────────────
evaluator = BasicAgentEvaluator(pass_threshold=0.7)

table = Table(title="Basic Agent Evaluation Results", show_header=True, header_style="bold magenta")
table.add_column("TC"); table.add_column("Metric"); table.add_column("Score"); 
table.add_column("Pass?"); table.add_column("Reason")

all_scores = {}
for tc in test_cases:
    results = evaluator.evaluate(tc, actual_steps=3, optimal_steps=2)
    for r in results:
        color = "green" if r.passed else "red"
        table.add_row(
            tc.test_id, r.metric,
            f"[{color}]{r.score:.3f}[/{color}]",
            f"[{color}]{'✅' if r.passed else '❌'}[/{color}]",
            r.reason[:60]
        )
        all_scores.setdefault(r.metric, []).append(r.score)

console.print(table)

# Summary
rprint("\n[bold]Overall Metric Averages:[/bold]")
for metric, scores in all_scores.items():
    avg = np.mean(scores)
    color = "green" if avg >= 0.7 else "red"
    rprint(f"  {metric:25s} [{color}]{avg:.3f}[/{color}]")


                                  Basic Agent Evaluation Results                                   
┏━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ TC     ┃ Metric           ┃ Score ┃ Pass? ┃ Reason                                              ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ TC-001 │ keyword_coverage │ 0.250 │ ❌    │ 1/4 keywords matched                                │
│ TC-001 │ tool_accuracy    │ 0.667 │ ❌    │ F1=0.67 | missing=set() | extra={'web_search_mock'} │
│ TC-001 │ step_efficiency  │ 0.667 │ ❌    │ Used 3 steps (optimal: 2). Ratio: 0.67              │
│ TC-001 │ latency_check    │ 1.000 │ ✅    │ Latency: 280ms (SLA: 5000ms)                        │
│ TC-002 │ keyword_coverage │ 1.000 │ ✅    │ 6/6 keywords matched                                │
│ TC-002 │ tool_accuracy    │ 1.000 │ ✅    │ F1=1.00 | missing=set() | extra=set()               │
│ TC-002 │ step_efficiency  │ 0.667 │ ❌    │ Used 3 steps (optimal: 2). Ratio: 0.67              │
│ TC-002 │ latency_check    │ 1.000 │ ✅    │ Latency: 150ms (SLA: 5000ms)                        │
│ TC-003 │ keyword_coverage │ 0.000 │ ❌    │ 0/3 keywords matched                                │
│ TC-003 │ tool_accuracy    │ 0.000 │ ❌    │ F1=0.00 | missing={'get_weather'} | extra=set()     │
│ TC-003 │ step_efficiency  │ 0.667 │ ❌    │ Used 3 steps (optimal: 2). Ratio: 0.67              │
│ TC-003 │ latency_check    │ 0.700 │ ❌    │ Latency: 6500ms (SLA: 5000ms)                       │
└────────┴──────────────────┴───────┴───────┴─────────────────────────────────────────────────────┘

Overall Metric Averages:

keyword_coverage          0.417

tool_accuracy             0.556

step_efficiency           0.667

latency_check             0.900

---
## 🧩 Section 3: Complex Single Agents

We now build sophisticated single-agent architectures:

1. **Planning Agent** — explicit plan → execute → reflect loop
2. **Reflection Agent** — self-critique and revision
3. **RAG Agent** — retrieval-augmented generation with evaluation
4. **Memory Agent** — semantic memory management with episodic + semantic stores

These patterns are the building blocks for production agents.


In [8]:
# ─── 3.1 PLANNING AGENT (Plan → Execute → Synthesize) ────────────────────────
# Classic pattern: first generate a plan, then execute each step, then synthesize.
# Useful for: multi-step tasks, document analysis, research workflows

class PlanExecuteState(TypedDict):
    messages: Annotated[list, add_messages]
    plan: List[str]          # ← ordered list of steps
    current_step: int         # ← pointer into plan
    step_results: List[str]   # ← accumulated results
    final_answer: str
    task: str


PLAN_LLM_RESPONSES = [
    json.dumps({
        "plan": [
            "Step 1: Identify key metrics for agent evaluation",
            "Step 2: Research industry benchmarks",
            "Step 3: Assess tooling options (DeepEval, RAGAS, custom)",
            "Step 4: Synthesize recommendations",
        ]
    }),
]

EXECUTOR_RESPONSES = [
    "Key metrics include: accuracy, tool_use_correctness, faithfulness, latency, safety_score.",
    "Industry benchmarks: AgentBench (multi-task), GAIA (real-world), WebArena (web tasks).",
    "Tooling: DeepEval excels at unit-test-style evaluation; RAGAS specializes in RAG pipelines; custom evaluators offer most flexibility.",
    "Recommendation: Use DeepEval for component testing + RAGAS for RAG quality + custom for domain-specific metrics.",
]


def planner_node(state: PlanExecuteState) -> dict:
    """Generate an execution plan from the task."""
    plan_llm = MockLLM(responses=PLAN_LLM_RESPONSES)
    response = plan_llm.invoke([HumanMessage(content=f"Create a plan for: {state['task']}")])
    
    try:
        parsed = json.loads(response.content)
        plan = parsed["plan"]
    except:
        plan = [response.content]
    
    return {
        "plan": plan,
        "current_step": 0,
        "step_results": [],
        "messages": [AIMessage(content=f"Plan created with {len(plan)} steps: {plan}")],
    }


def executor_node(state: PlanExecuteState) -> dict:
    """Execute the current step of the plan."""
    step_idx = state["current_step"]
    if step_idx >= len(state["plan"]):
        return {}
    
    current_step = state["plan"][step_idx]
    exec_llm = MockLLM(responses=EXECUTOR_RESPONSES)
    response = exec_llm.invoke([HumanMessage(content=f"Execute: {current_step}")])
    
    new_results = state["step_results"] + [response.content]
    
    return {
        "step_results": new_results,
        "current_step": step_idx + 1,
        "messages": [AIMessage(content=f"Step {step_idx+1} done: {response.content[:80]}...")],
    }


def synthesizer_node(state: PlanExecuteState) -> dict:
    """Combine all step results into a final answer."""
    synthesis = "\n".join([f"• {r}" for r in state["step_results"]])
    final = f"Task: {state['task']}\n\nFindings:\n{synthesis}"
    
    return {
        "final_answer": final,
        "messages": [AIMessage(content=final)],
    }


def plan_execute_router(state: PlanExecuteState) -> Literal["execute", "synthesize"]:
    if state["current_step"] < len(state["plan"]):
        return "execute"
    return "synthesize"


def build_plan_execute_agent():
    builder = StateGraph(PlanExecuteState)
    builder.add_node("plan", planner_node)
    builder.add_node("execute", executor_node)
    builder.add_node("synthesize", synthesizer_node)
    
    builder.add_edge(START, "plan")
    builder.add_conditional_edges("plan", lambda s: "execute", {"execute": "execute"})
    builder.add_conditional_edges("execute", plan_execute_router, {
        "execute": "execute",
        "synthesize": "synthesize",
    })
    builder.add_edge("synthesize", END)
    
    return builder.compile()


plan_agent = build_plan_execute_agent()
plan_result = plan_agent.invoke({
    "messages": [],
    "plan": [], "current_step": 0, "step_results": [], "final_answer": "",
    "task": "Build a comprehensive agent evaluation framework",
})

rprint(Panel(
    f"[bold]Task:[/bold] {plan_result['task']}\n\n"
    f"[bold]Plan ({len(plan_result['plan'])} steps):[/bold]\n" +
    "\n".join(f"  {i+1}. {s}" for i, s in enumerate(plan_result["plan"])) +
    f"\n\n[bold green]Final Answer (first 200 chars):[/bold green]\n{plan_result['final_answer'][:200]}...",
    title="[cyan]3.1 Plan-Execute Agent[/cyan]",
    border_style="cyan"
))


╭──────────────────────────────────────────── 3.1 Plan-Execute Agent ─────────────────────────────────────────────╮
│ Task: Build a comprehensive agent evaluation framework                                                          │
│                                                                                                                 │
│ Plan (4 steps):                                                                                                 │
│   1. Step 1: Identify key metrics for agent evaluation                                                          │
│   2. Step 2: Research industry benchmarks                                                                       │
│   3. Step 3: Assess tooling options (DeepEval, RAGAS, custom)                                                   │
│   4. Step 4: Synthesize recommendations                                                                         │
│                                                                                                                 │
│ Final Answer (first 200 chars):                                                                                 │
│ Task: Build a comprehensive agent evaluation framework                                                          │
│                                                                                                                 │
│ Findings:                                                                                                       │
│ • Key metrics include: accuracy, tool_use_correctness, faithfulness, latency, safety_score.                     │
│ • Key metrics include: accuracy, tool_use_...                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [9]:
# ─── 3.2 REFLECTION AGENT (Self-Critique + Revision) ─────────────────────────
# The agent generates a draft, then critiques it, then revises based on critique.
# Pattern: generate → reflect → [revise if needed] → END
# This is a key pattern for high-quality outputs.

class ReflectionState(TypedDict):
    messages: Annotated[list, add_messages]
    draft: str
    critique: str
    revision: str
    iteration: int
    max_iterations: int
    quality_score: float   # ← agent self-rates its output


DRAFT_RESPONSES = [
    "Draft v1: Agent evaluation should include accuracy and latency metrics.",
    "Draft v2: Agent evaluation requires a comprehensive framework covering correctness, tool use, faithfulness, latency, and safety. Each dimension needs specific metrics and test cases.",
]

CRITIQUE_RESPONSES = [
    json.dumps({
        "issues": ["Too brief", "Missing tool evaluation", "No mention of safety"],
        "quality_score": 0.4,
        "should_revise": True,
        "suggestion": "Expand with concrete metrics for each dimension. Add safety and tool accuracy.",
    }),
    json.dumps({
        "issues": [],
        "quality_score": 0.85,
        "should_revise": False,
        "suggestion": "Good. Maybe add examples of evaluation frameworks.",
    }),
]

REVISION_RESPONSES = [
    "Revised: Agent evaluation requires a comprehensive framework covering: (1) Correctness — semantic similarity to ground truth; (2) Tool Use — precision/recall of tool selection; (3) Faithfulness — outputs grounded in retrieved context; (4) Latency — response time under SLA; (5) Safety — no harmful outputs. Frameworks: DeepEval for unit tests, RAGAS for RAG pipelines.",
]


def generator_node(state: ReflectionState) -> dict:
    idx = state.get("iteration", 0)
    llm = MockLLM(responses=DRAFT_RESPONSES)
    response = llm.invoke(state["messages"])
    draft = DRAFT_RESPONSES[min(idx, len(DRAFT_RESPONSES)-1)]
    
    return {
        "draft": draft,
        "messages": [AIMessage(content=f"[Draft v{idx+1}]: {draft}")],
    }


def critic_node(state: ReflectionState) -> dict:
    idx = state.get("iteration", 0)
    critique_json = CRITIQUE_RESPONSES[min(idx, len(CRITIQUE_RESPONSES)-1)]
    critique_data = json.loads(critique_json)
    
    critique = f"Issues: {critique_data['issues']}. Quality: {critique_data['quality_score']}. {critique_data['suggestion']}"
    
    return {
        "critique": critique,
        "quality_score": critique_data["quality_score"],
        "iteration": idx + 1,
        "messages": [AIMessage(content=f"[Critique]: {critique}")],
    }


def reviser_node(state: ReflectionState) -> dict:
    revision = REVISION_RESPONSES[0]
    return {
        "revision": revision,
        "draft": revision,  # ← update draft with revision
        "messages": [AIMessage(content=f"[Revision]: {revision}")],
    }


def reflection_router(state: ReflectionState) -> Literal["revise", "end"]:
    """Continue revising if quality below threshold and iterations remaining."""
    max_iter = state.get("max_iterations", 3)
    current_iter = state.get("iteration", 0)
    quality = state.get("quality_score", 0.0)
    
    if current_iter < max_iter and quality < 0.8:
        return "revise"
    return "end"


def build_reflection_agent():
    builder = StateGraph(ReflectionState)
    builder.add_node("generate", generator_node)
    builder.add_node("critique", critic_node)
    builder.add_node("revise", reviser_node)
    
    builder.add_edge(START, "generate")
    builder.add_edge("generate", "critique")
    builder.add_conditional_edges("critique", reflection_router, {
        "revise": "revise",
        "end": END,
    })
    builder.add_edge("revise", "critique")  # ← loop back
    
    return builder.compile()


reflection_agent = build_reflection_agent()
reflection_result = reflection_agent.invoke({
    "messages": [HumanMessage(content="Write about agent evaluation")],
    "draft": "", "critique": "", "revision": "",
    "iteration": 0, "max_iterations": 3, "quality_score": 0.0,
})

rprint(Panel(
    f"[bold]Final Quality Score:[/bold] {reflection_result['quality_score']:.2f}\n"
    f"[bold]Iterations:[/bold] {reflection_result['iteration']}\n"
    f"[bold]Last Critique:[/bold] {reflection_result['critique'][:100]}...\n\n"
    f"[bold green]Final Draft:[/bold green]\n{reflection_result['draft'][:300]}...",
    title="[cyan]3.2 Reflection Agent — Self-Critique Loop[/cyan]",
    border_style="cyan"
))

# ── Evaluate Reflection Quality ───────────────────────────────────────────────
rprint("\n[bold]Evaluating Reflection Agent:[/bold]")
reflection_scores = {
    "quality_improvement": reflection_result["quality_score"],
    "iterations_used": reflection_result["iteration"],
    "efficiency": max(0, 1 - (reflection_result["iteration"] - 1) * 0.2),  # penalize excessive loops
}
for k, v in reflection_scores.items():
    rprint(f"  {k:25s}: {v:.3f}")


╭─────────────────────────────────── 3.2 Reflection Agent — Self-Critique Loop ───────────────────────────────────╮
│ Final Quality Score: 0.85                                                                                       │
│ Iterations: 2                                                                                                   │
│ Last Critique: Issues: []. Quality: 0.85. Good. Maybe add examples of evaluation frameworks....                 │
│                                                                                                                 │
│ Final Draft:                                                                                                    │
│ Revised: Agent evaluation requires a comprehensive framework covering: (1) Correctness — semantic similarity to │
│ ground truth; (2) Tool Use — precision/recall of tool selection; (3) Faithfulness — outputs grounded in         │
│ retrieved context; (4) Latency — response time under SLA; (5) Safety — no harmful ou...                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Evaluating Reflection Agent:

quality_improvement      : 0.850

iterations_used          : 2.000

efficiency               : 0.800

In [10]:
# ─── 3.3 RAG AGENT (Retrieval-Augmented Generation) ──────────────────────────
# RAG = retrieve relevant docs → augment prompt → generate answer
# Critical for evaluation: RAGAS was designed specifically for RAG pipelines

# ── Tiny in-memory vector store (cosine similarity on TF-IDF-like vectors) ────
class TinyVectorStore:
    """
    Minimal vector store using keyword overlap as 'similarity'.
    Production: replace with FAISS, ChromaDB, Pinecone, etc.
    """
    def __init__(self):
        self.docs: List[Dict] = []
    
    def add(self, text: str, metadata: dict = None):
        words = set(text.lower().split())
        self.docs.append({
            "text": text,
            "words": words,
            "metadata": metadata or {},
            "id": len(self.docs),
        })
    
    def similarity(self, query_words: set, doc_words: set) -> float:
        """Jaccard similarity"""
        if not query_words or not doc_words:
            return 0.0
        return len(query_words & doc_words) / len(query_words | doc_words)
    
    def retrieve(self, query: str, k: int = 3) -> List[Dict]:
        query_words = set(query.lower().split())
        scored = [
            (self.similarity(query_words, doc["words"]), doc)
            for doc in self.docs
        ]
        scored.sort(key=lambda x: x[0], reverse=True)
        return [(score, doc) for score, doc in scored[:k] if score > 0]


# ── Knowledge base ────────────────────────────────────────────────────────────
KNOWLEDGE_BASE = [
    ("Agent evaluation is the process of measuring agent performance across multiple dimensions including correctness, tool use accuracy, faithfulness to retrieved context, latency, and safety.", {"topic": "evaluation", "source": "textbook"}),
    ("DeepEval is an open-source LLM evaluation framework that provides unit-test-style evaluation for LLM outputs including hallucination detection, answer relevancy, and contextual precision.", {"topic": "deepeval", "source": "docs"}),
    ("RAGAS (Retrieval Augmented Generation Assessment) evaluates RAG pipelines on faithfulness, answer relevancy, context precision, and context recall metrics.", {"topic": "ragas", "source": "docs"}),
    ("LangGraph enables building stateful, multi-actor LLM applications as graphs with nodes representing agent steps and edges representing transitions between steps.", {"topic": "langgraph", "source": "docs"}),
    ("Tool calling in LLM agents involves structured function invocation where the model outputs a JSON specification of which tool to call with what arguments.", {"topic": "tools", "source": "docs"}),
    ("Multi-agent systems consist of multiple specialized agents that collaborate, with patterns including supervisor-worker, peer-to-peer, and sequential handoff.", {"topic": "multi-agent", "source": "research"}),
    ("Model Context Protocol (MCP) is an open standard for connecting AI models to data sources and tools using a client-server architecture.", {"topic": "mcp", "source": "anthropic-docs"}),
    ("Faithfulness measures whether the agent's answer is grounded in the retrieved context. An unfaithful answer introduces information not present in the context.", {"topic": "faithfulness", "source": "ragas"}),
]

vectorstore = TinyVectorStore()
for text, meta in KNOWLEDGE_BASE:
    vectorstore.add(text, meta)

rprint(f"[green]Knowledge base loaded: {len(vectorstore.docs)} documents[/green]")


# ── RAG Agent State ───────────────────────────────────────────────────────────
class RAGState(TypedDict):
    messages: Annotated[list, add_messages]
    query: str
    retrieved_docs: List[Dict]   # ← for evaluation
    context: str
    answer: str
    sources: List[str]


RAG_ANSWER_RESPONSES = [
    "Based on the retrieved context: {context_placeholder}. This answers your question about {query_placeholder}.",
]


def retriever_node(state: RAGState) -> dict:
    """Retrieve relevant documents for the query."""
    results = vectorstore.retrieve(state["query"], k=3)
    
    retrieved = []
    for score, doc in results:
        retrieved.append({
            "text": doc["text"],
            "score": score,
            "source": doc["metadata"].get("source", "unknown"),
            "topic": doc["metadata"].get("topic", "general"),
        })
    
    context = "\n\n".join([f"[Doc {i+1}]: {d['text']}" for i, d in enumerate(retrieved)])
    sources = [d["source"] for d in retrieved]
    
    return {
        "retrieved_docs": retrieved,
        "context": context,
        "sources": sources,
        "messages": [AIMessage(content=f"Retrieved {len(retrieved)} relevant documents.")],
    }


def rag_generator_node(state: RAGState) -> dict:
    """Generate answer grounded in retrieved context."""
    # In production: use LLM with context in prompt
    # Here: deterministic extraction from context
    
    context = state.get("context", "")
    query = state.get("query", "")
    
    # Simple extraction: find most relevant sentence
    sentences = context.split(".")
    query_words = set(query.lower().split())
    best_sentences = []
    
    for sent in sentences:
        sent = sent.strip()
        if len(sent) < 20:
            continue
        overlap = len(query_words & set(sent.lower().split()))
        if overlap >= 2:
            best_sentences.append(sent)
    
    if best_sentences:
        answer = " ".join(best_sentences[:2]) + "."
    else:
        answer = f"Based on the context: {context[:200]}."
    
    return {
        "answer": answer,
        "messages": [AIMessage(content=answer)],
    }


def build_rag_agent():
    builder = StateGraph(RAGState)
    builder.add_node("retrieve", retriever_node)
    builder.add_node("generate", rag_generator_node)
    
    builder.add_edge(START, "retrieve")
    builder.add_edge("retrieve", "generate")
    builder.add_edge("generate", END)
    
    return builder.compile()


rag_agent = build_rag_agent()

# Run several queries
queries = [
    "What is RAGAS and how does it evaluate RAG systems?",
    "How does LangGraph work for building agents?",
    "What is faithfulness in agent evaluation?",
]

rag_results = []
for q in queries:
    result = rag_agent.invoke({
        "messages": [HumanMessage(content=q)],
        "query": q,
        "retrieved_docs": [], "context": "", "answer": "", "sources": [],
    })
    rag_results.append(result)

table = Table(title="RAG Agent Results", header_style="bold cyan")
table.add_column("Query", max_width=35)
table.add_column("Docs Retrieved")
table.add_column("Top Source")
table.add_column("Answer (truncated)", max_width=50)

for result in rag_results:
    table.add_row(
        result["query"][:33] + "...",
        str(len(result["retrieved_docs"])),
        result["retrieved_docs"][0]["topic"] if result["retrieved_docs"] else "none",
        result["answer"][:48] + "..." if len(result["answer"]) > 48 else result["answer"],
    )

console.print(table)


Knowledge base loaded: 8 documents

                                                 RAG Agent Results                                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Query                               ┃ Docs Retrieved ┃ Top Source   ┃ Answer (truncated)                        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ What is RAGAS and how does it       │ 3              │ ragas        │ [Doc 1]: RAGAS (Retrieval Augmented       │
│ eva...                              │                │              │ Generation A...                           │
│ How does LangGraph work for         │ 3              │ langgraph    │ [Doc 1]: LangGraph enables building       │
│ build...                            │                │              │ stateful, mu...                           │
│ What is faithfulness in agent       │ 3              │ faithfulness │ [Doc 1]: Faithfulness measures whether    │
│ eva...                              │                │              │ the agent...                              │
└─────────────────────────────────────┴────────────────┴──────────────┴───────────────────────────────────────────┘

---
## 🤝 Section 4: Multi-Agent Systems

Single agents break down for complex tasks. Multi-agent systems solve this through:

- **Supervisor Pattern** — one orchestrator delegates to specialist workers
- **Subgraph Pattern** — agents are embedded as subgraphs
- **Peer-to-Peer Handoff** — agents pass control directly to each other
- **Shared Memory Pattern** — agents write to a common blackboard

```
SUPERVISOR PATTERN              HANDOFF PATTERN
                                
     Supervisor                 Agent A → Agent B → Agent C
    ↙    ↓    ↘                    ↑__________________|
 Worker Worker Worker
  (A)   (B)   (C)
```


In [11]:
# ─── 4.1 SUPERVISOR MULTI-AGENT SYSTEM ───────────────────────────────────────
# Supervisor receives task, routes to specialist workers, collects results.

# ── Worker agents (specialists) ───────────────────────────────────────────────
@tool
def research_tool(topic: str) -> str:
    """Research a topic and return findings."""
    findings = {
        "evaluation": "Evaluation frameworks include DeepEval, RAGAS, and custom Python evaluators.",
        "langgraph": "LangGraph provides StateGraph with nodes, edges, and checkpointers.",
        "mcp": "MCP enables standardized tool connections between agents and external services.",
        "multi-agent": "Multi-agent systems use supervisor, peer, and subgraph patterns.",
    }
    for key, val in findings.items():
        if key in topic.lower():
            return val
    return f"Research on '{topic}': Found general knowledge applicable to this topic."


@tool
def analysis_tool(data: str) -> str:
    """Analyze data and produce insights."""
    word_count = len(data.split())
    keywords = [w for w in data.split() if len(w) > 5][:5]
    return f"Analysis: {word_count} words. Key themes: {', '.join(set(keywords))}. Sentiment: Informative."


@tool  
def writer_tool(content: str) -> str:
    """Take research/analysis and write a professional summary."""
    sentences = content.split(".")
    summary = f"Executive Summary: {sentences[0].strip()}. This has significant implications for practitioners."
    return summary


# ── Multi-Agent State ─────────────────────────────────────────────────────────
class MultiAgentState(TypedDict):
    messages: Annotated[list, add_messages]
    task: str
    worker_outputs: Dict[str, str]   # ← tracks each worker's contribution
    next_worker: str                  # ← supervisor routing decision
    final_report: str
    supervisor_iterations: int


SUPERVISOR_ROUTING = ["researcher", "analyst", "writer", "FINISH"]


def supervisor_node(state: MultiAgentState) -> dict:
    """
    Orchestrator: decides which worker to delegate to next.
    In production: LLM decides based on task and current state.
    """
    iteration = state.get("supervisor_iterations", 0)
    
    if iteration < len(SUPERVISOR_ROUTING):
        next_w = SUPERVISOR_ROUTING[iteration]
    else:
        next_w = "FINISH"
    
    return {
        "next_worker": next_w,
        "supervisor_iterations": iteration + 1,
        "messages": [AIMessage(content=f"[Supervisor] → delegating to: {next_w}")],
    }


def researcher_node(state: MultiAgentState) -> dict:
    """Specialist: research agent"""
    result = research_tool.invoke({"topic": state["task"]})
    return {
        "worker_outputs": {**state.get("worker_outputs", {}), "researcher": result},
        "messages": [AIMessage(content=f"[Researcher]: {result}")],
    }


def analyst_node(state: MultiAgentState) -> dict:
    """Specialist: analysis agent"""
    research = state.get("worker_outputs", {}).get("researcher", state["task"])
    result = analysis_tool.invoke({"data": research})
    return {
        "worker_outputs": {**state.get("worker_outputs", {}), "analyst": result},
        "messages": [AIMessage(content=f"[Analyst]: {result}")],
    }


def writer_node(state: MultiAgentState) -> dict:
    """Specialist: writing agent"""
    all_context = " ".join(state.get("worker_outputs", {}).values())
    result = writer_tool.invoke({"content": all_context})
    return {
        "worker_outputs": {**state.get("worker_outputs", {}), "writer": result},
        "final_report": result,
        "messages": [AIMessage(content=f"[Writer]: {result}")],
    }


def supervisor_router(state: MultiAgentState) -> str:
    next_w = state.get("next_worker", "FINISH")
    if next_w == "FINISH" or next_w not in ["researcher", "analyst", "writer"]:
        return "end"
    return next_w


def build_supervisor_system():
    builder = StateGraph(MultiAgentState)
    
    # Add all nodes
    builder.add_node("supervisor", supervisor_node)
    builder.add_node("researcher", researcher_node)
    builder.add_node("analyst", analyst_node)
    builder.add_node("writer", writer_node)
    
    # Entry point
    builder.add_edge(START, "supervisor")
    
    # Supervisor routes to workers
    builder.add_conditional_edges("supervisor", supervisor_router, {
        "researcher": "researcher",
        "analyst": "analyst",
        "writer": "writer",
        "end": END,
    })
    
    # All workers report back to supervisor
    builder.add_edge("researcher", "supervisor")
    builder.add_edge("analyst", "supervisor")
    builder.add_edge("writer", "supervisor")
    
    return builder.compile()


supervisor_system = build_supervisor_system()
multi_result = supervisor_system.invoke({
    "messages": [HumanMessage(content="Build a report on agent evaluation frameworks")],
    "task": "agent evaluation frameworks",
    "worker_outputs": {},
    "next_worker": "",
    "final_report": "",
    "supervisor_iterations": 0,
})

rprint(Panel(
    "[bold]Multi-Agent Execution:[/bold]\n\n" +
    "\n".join([
        f"  [bold]{k.upper():12s}[/bold]: {v[:80]}..." 
        for k, v in multi_result["worker_outputs"].items()
    ]) +
    f"\n\n[bold green]Final Report:[/bold green]\n{multi_result['final_report']}",
    title="[cyan]4.1 Supervisor Multi-Agent System[/cyan]",
    border_style="cyan"
))
rprint(f"[green]✅ Supervisor used {multi_result['supervisor_iterations']} routing decisions[/green]")


╭─────────────────────────────────────── 4.1 Supervisor Multi-Agent System ───────────────────────────────────────╮
│ Multi-Agent Execution:                                                                                          │
│                                                                                                                 │
│   RESEARCHER  : Evaluation frameworks include DeepEval, RAGAS, and custom Python evaluators....                 │
│   ANALYST     : Analysis: 9 words. Key themes: Evaluation, DeepEval,, RAGAS,, include, framework...             │
│   WRITER      : Executive Summary: Evaluation frameworks include DeepEval, RAGAS, and custom Pyt...             │
│                                                                                                                 │
│ Final Report:                                                                                                   │
│ Executive Summary: Evaluation frameworks include DeepEval, RAGAS, and custom Python evaluators. This has        │
│ significant implications for practitioners.                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✅ Supervisor used 4 routing decisions

In [12]:
# ─── 4.2 SUBGRAPH PATTERN ─────────────────────────────────────────────────────
# Compose complex agents by nesting graphs inside graphs.
# Inner graph = subgraph; outer graph = orchestrator.
# Useful for: reusable agent modules, independent testing of sub-components.

# ── Inner subgraph: document processing pipeline ──────────────────────────────
class DocProcessState(TypedDict):
    raw_text: str
    cleaned_text: str
    summary: str
    entities: List[str]


def cleaner_node(state: DocProcessState) -> dict:
    """Clean raw text"""
    cleaned = re.sub(r'\s+', ' ', state["raw_text"]).strip()
    cleaned = cleaned.lower()
    return {"cleaned_text": cleaned}


def entity_extractor_node(state: DocProcessState) -> dict:
    """Extract named entities (mock: just extract capitalized words from original)"""
    words = state["raw_text"].split()
    entities = [w for w in words if w[0].isupper() and len(w) > 3][:5]
    return {"entities": entities}


def summarizer_subgraph_node(state: DocProcessState) -> dict:
    """Summarize the document"""
    text = state.get("cleaned_text", state["raw_text"])
    sentences = text.split(".")
    summary = ". ".join(sentences[:2]).strip() + "."
    return {"summary": summary}


# Build the subgraph
doc_subgraph_builder = StateGraph(DocProcessState)
doc_subgraph_builder.add_node("clean", cleaner_node)
doc_subgraph_builder.add_node("extract_entities", entity_extractor_node)
doc_subgraph_builder.add_node("summarize", summarizer_subgraph_node)
doc_subgraph_builder.add_edge(START, "clean")
doc_subgraph_builder.add_edge("clean", "extract_entities")
doc_subgraph_builder.add_edge("extract_entities", "summarize")
doc_subgraph_builder.add_edge("summarize", END)
doc_subgraph = doc_subgraph_builder.compile()

# ── Outer orchestrator graph ───────────────────────────────────────────────────
class OrchestratorState(TypedDict):
    messages: Annotated[list, add_messages]
    documents: List[str]
    processed_docs: List[Dict]
    current_doc_idx: int
    final_synthesis: str


def doc_processor_node(state: OrchestratorState) -> dict:
    """Use the subgraph to process the current document"""
    idx = state.get("current_doc_idx", 0)
    docs = state["documents"]
    
    if idx >= len(docs):
        return {}
    
    raw_text = docs[idx]
    # ← INVOKE SUBGRAPH HERE
    sub_result = doc_subgraph.invoke({"raw_text": raw_text, "cleaned_text": "", "summary": "", "entities": []})
    
    processed = state.get("processed_docs", []) + [{
        "doc_id": idx,
        "summary": sub_result["summary"],
        "entities": sub_result["entities"],
    }]
    
    return {
        "processed_docs": processed,
        "current_doc_idx": idx + 1,
        "messages": [AIMessage(content=f"Processed doc {idx+1}: {sub_result['summary'][:60]}...")],
    }


def synthesizer_orch_node(state: OrchestratorState) -> dict:
    summaries = [d["summary"] for d in state.get("processed_docs", [])]
    all_entities = [e for d in state.get("processed_docs", []) for e in d.get("entities", [])]
    synthesis = f"Processed {len(summaries)} documents. Key entities: {', '.join(set(all_entities)[:8])}."
    return {
        "final_synthesis": synthesis,
        "messages": [AIMessage(content=synthesis)],
    }


def orch_router(state: OrchestratorState) -> Literal["process_doc", "synthesize"]:
    if state.get("current_doc_idx", 0) < len(state["documents"]):
        return "process_doc"
    return "synthesize"


orch_builder = StateGraph(OrchestratorState)
orch_builder.add_node("process_doc", doc_processor_node)
orch_builder.add_node("synthesize", synthesizer_orch_node)
orch_builder.add_edge(START, "process_doc")
orch_builder.add_conditional_edges("process_doc", orch_router, {
    "process_doc": "process_doc",
    "synthesize": "synthesize",
})
orch_builder.add_edge("synthesize", END)
orchestrator = orch_builder.compile()


# ── Run with 3 documents ──────────────────────────────────────────────────────
sample_docs = [
    "DeepEval is an Open Source framework developed by Confident AI. It provides LLM unit testing for evaluating outputs.",
    "RAGAS was created by Shahul Es and Jithin James. It focuses on Retrieval Augmented Generation evaluation pipelines.",
    "LangGraph from LangChain enables stateful agent graphs. Harrison Chase founded LangChain in 2022.",
]

orch_result = orchestrator.invoke({
    "messages": [],
    "documents": sample_docs,
    "processed_docs": [],
    "current_doc_idx": 0,
    "final_synthesis": "",
})

rprint(Panel(
    f"[bold]Documents processed:[/bold] {len(orch_result['processed_docs'])}\n\n" +
    "\n".join([
        f"  Doc {d['doc_id']+1}: {d['summary'][:60]}... | entities: {d['entities']}"
        for d in orch_result["processed_docs"]
    ]) +
    f"\n\n[bold green]Synthesis:[/bold green] {orch_result['final_synthesis']}",
    title="[cyan]4.2 Subgraph Pattern — Nested Graphs[/cyan]",
    border_style="cyan"
))


TypeError: 'set' object is not subscriptable

In [ ]:
# ─── 4.3 PEER-TO-PEER HANDOFF WITH Command ───────────────────────────────────
# Agents transfer control explicitly via Command objects.
# Each agent can modify state AND specify the next node in one return.

from langgraph.types import Command

class HandoffState(TypedDict):
    messages: Annotated[list, add_messages]
    task_type: str          # "research" | "code" | "review"
    context: str
    handoff_history: List[str]


def triage_agent(state: HandoffState) -> Command:
    """Entry point: classify task and route"""
    task = state["messages"][-1].content.lower()
    
    if any(w in task for w in ["research", "find", "search", "what is"]):
        next_agent = "research_agent"
        task_type = "research"
    elif any(w in task for w in ["code", "write", "implement", "build"]):
        next_agent = "coding_agent"
        task_type = "code"
    else:
        next_agent = "generalist_agent"
        task_type = "general"
    
    return Command(
        goto=next_agent,   # ← explicit routing via Command
        update={
            "task_type": task_type,
            "handoff_history": state.get("handoff_history", []) + [f"triage→{next_agent}"],
            "messages": [AIMessage(content=f"[Triage] Task type: {task_type}. Routing to {next_agent}.")],
        }
    )


def research_agent_h(state: HandoffState) -> Command:
    """Research specialist"""
    task = state["messages"][0].content
    result = f"Research findings on '{task[:40]}': This is an important topic with multiple dimensions."
    
    # After research, hand off to review_agent
    return Command(
        goto="review_agent",
        update={
            "context": result,
            "handoff_history": state.get("handoff_history", []) + ["research→review"],
            "messages": [AIMessage(content=f"[Research Agent]: {result}")],
        }
    )


def coding_agent_h(state: HandoffState) -> Command:
    """Code specialist"""
    task = state["messages"][0].content
    result = f"```python\n# Implementation for: {task[:40]}\ndef solution():\n    return 'Implemented'\n```"
    
    return Command(
        goto="review_agent",
        update={
            "context": result,
            "handoff_history": state.get("handoff_history", []) + ["coding→review"],
            "messages": [AIMessage(content=f"[Coding Agent]: {result}")],
        }
    )


def generalist_agent_h(state: HandoffState) -> Command:
    task = state["messages"][0].content
    result = f"General response to: {task[:40]}. Providing a comprehensive overview."
    
    return Command(
        goto="review_agent",
        update={
            "context": result,
            "handoff_history": state.get("handoff_history", []) + ["generalist→review"],
            "messages": [AIMessage(content=f"[Generalist Agent]: {result}")],
        }
    )


def review_agent_h(state: HandoffState) -> dict:
    """Final reviewer — terminates the chain"""
    context = state.get("context", "")
    review = f"✅ Reviewed and approved. Task type: {state['task_type']}. Output quality: High. History: {' → '.join(state.get('handoff_history', []))}"
    
    return {
        "messages": [AIMessage(content=review)],
        "context": review,
    }


def build_handoff_system():
    builder = StateGraph(HandoffState)
    builder.add_node("triage_agent", triage_agent)
    builder.add_node("research_agent", research_agent_h)
    builder.add_node("coding_agent", coding_agent_h)
    builder.add_node("generalist_agent", generalist_agent_h)
    builder.add_node("review_agent", review_agent_h)
    
    builder.add_edge(START, "triage_agent")
    builder.add_edge("review_agent", END)
    
    return builder.compile()


handoff_system = build_handoff_system()

# Test with different task types
test_tasks = [
    "Research the best practices for LLM agent evaluation",
    "Write Python code for a RAG pipeline",
    "What should I have for lunch?",
]

rprint(Panel("[bold]4.3 Peer-to-Peer Handoff with Command[/bold]", border_style="cyan"))
for task in test_tasks:
    result = handoff_system.invoke({
        "messages": [HumanMessage(content=task)],
        "task_type": "",
        "context": "",
        "handoff_history": [],
    })
    route = " → ".join(result["handoff_history"])
    rprint(f"  [bold]Task:[/bold] {task[:50]}...")
    rprint(f"  [yellow]Route:[/yellow] {route}")
    rprint(f"  [green]Result:[/green] {result['messages'][-1].content[:80]}...")
    rprint()


---
## 🔌 Section 5: MCP & Advanced Tool Calling

**Model Context Protocol (MCP)** is an open standard by Anthropic that defines how AI models connect to external data sources and tools. Think of it as "USB for AI agents."

```
┌─────────────────────────────────────────────────────────────────┐
│                    MCP ARCHITECTURE                              │
│                                                                  │
│  Agent (MCP Client)  ←──── MCP Protocol ────→  MCP Server      │
│         │                                            │           │
│    [LangGraph]         JSON-RPC over SSE/stdio   [Tool Host]    │
│                                                   (DB, API,     │
│                                                    FileSystem)   │
└─────────────────────────────────────────────────────────────────┘
```

### MCP Capabilities
| Capability | What it provides |
|------------|-----------------|
| **Resources** | Expose data as context (files, DB rows, API results) |
| **Tools** | Callable functions the model can invoke |
| **Prompts** | Reusable prompt templates |
| **Sampling** | LLM completion requests |

### Why MCP Matters for Evaluation
- Tools are discoverable at runtime → test suite must be tool-aware
- Tool calls are structured JSON → easy to evaluate structurally  
- MCP servers are stateless → reproducible test scenarios


In [13]:
# ─── 5.1 MCP SERVER SIMULATION ────────────────────────────────────────────────
# We implement the MCP client-server pattern in pure Python.
# In production: use mcp library (pip install mcp) with real servers.

# ── MCP Protocol Types ─────────────────────────────────────────────────────────
@dataclass
class MCPTool:
    name: str
    description: str
    input_schema: Dict
    
    def to_dict(self) -> Dict:
        return {
            "name": self.name,
            "description": self.description,
            "inputSchema": self.input_schema,
        }


@dataclass
class MCPResource:
    uri: str
    name: str
    description: str
    mime_type: str = "text/plain"


@dataclass
class MCPCallResult:
    content: List[Dict]
    is_error: bool = False
    
    @classmethod
    def success(cls, text: str) -> "MCPCallResult":
        return cls(content=[{"type": "text", "text": text}], is_error=False)
    
    @classmethod
    def error(cls, message: str) -> "MCPCallResult":
        return cls(content=[{"type": "text", "text": f"Error: {message}"}], is_error=True)


class MCPServer:
    """
    Simulated MCP Server.
    A real MCP server exposes tools via JSON-RPC over stdio or SSE.
    """
    def __init__(self, name: str, version: str = "1.0.0"):
        self.name = name
        self.version = version
        self._tools: Dict[str, MCPTool] = {}
        self._tool_handlers: Dict[str, Callable] = {}
        self._resources: Dict[str, MCPResource] = {}
        self._resource_data: Dict[str, str] = {}
        self.call_log: List[Dict] = []  # ← evaluation hook
    
    def register_tool(self, tool: MCPTool, handler: Callable):
        self._tools[tool.name] = tool
        self._tool_handlers[tool.name] = handler
    
    def register_resource(self, resource: MCPResource, data: str):
        self._resources[resource.uri] = resource
        self._resource_data[resource.uri] = data
    
    def list_tools(self) -> List[Dict]:
        """MCP tools/list"""
        return [t.to_dict() for t in self._tools.values()]
    
    def call_tool(self, name: str, arguments: Dict) -> MCPCallResult:
        """MCP tools/call"""
        if name not in self._tool_handlers:
            return MCPCallResult.error(f"Tool '{name}' not found")
        
        start = time.time()
        try:
            result = self._tool_handlers[name](**arguments)
            call_result = MCPCallResult.success(str(result))
        except Exception as e:
            call_result = MCPCallResult.error(str(e))
        
        self.call_log.append({
            "tool": name, "args": arguments,
            "result": call_result.content[0]["text"][:100],
            "latency_ms": (time.time() - start) * 1000,
            "error": call_result.is_error,
        })
        return call_result
    
    def read_resource(self, uri: str) -> str:
        return self._resource_data.get(uri, f"Resource not found: {uri}")


# ── Build a demo MCP Server (Insurance domain for Harsh's context) ─────────────
insurance_mcp = MCPServer("insurance-tools", "1.0.0")

# Register tools
insurance_mcp.register_tool(
    MCPTool(
        name="lookup_policy",
        description="Look up an insurance policy by policy number",
        input_schema={
            "type": "object",
            "properties": {"policy_id": {"type": "string", "description": "Policy number"}},
            "required": ["policy_id"]
        }
    ),
    handler=lambda policy_id: json.dumps({
        "policy_id": policy_id,
        "holder": "John Doe",
        "type": "Term Life",
        "premium": 12500,
        "status": "Active",
        "grievances": 0,
    })
)

insurance_mcp.register_tool(
    MCPTool(
        name="submit_grievance",
        description="Submit a grievance for a policy",
        input_schema={
            "type": "object",
            "properties": {
                "policy_id": {"type": "string"},
                "category": {"type": "string", "enum": ["claim_rejection", "delay", "mis_selling", "other"]},
                "description": {"type": "string"},
            },
            "required": ["policy_id", "category", "description"]
        }
    ),
    handler=lambda policy_id, category, description: json.dumps({
        "grievance_id": f"GR-{uuid.uuid4().hex[:6].upper()}",
        "status": "submitted",
        "policy_id": policy_id,
        "category": category,
        "eta_days": 15,
    })
)

insurance_mcp.register_tool(
    MCPTool(
        name="check_winnability",
        description="Predict whether a grievance is likely to be resolved in claimant's favor",
        input_schema={
            "type": "object",
            "properties": {
                "grievance_id": {"type": "string"},
                "evidence_summary": {"type": "string"},
            },
            "required": ["grievance_id", "evidence_summary"]
        }
    ),
    handler=lambda grievance_id, evidence_summary: json.dumps({
        "grievance_id": grievance_id,
        "winnability_score": round(random.uniform(0.4, 0.9), 2),
        "confidence": "Medium",
        "key_factors": ["documentation_completeness", "policy_terms", "regulatory_precedent"],
    })
)

# Register resources
insurance_mcp.register_resource(
    MCPResource("insurance://regulations/irdai-2023", "IRDAI Regulations 2023", "Insurance regulatory guidelines"),
    data="IRDAI Circular 2023: All insurers must resolve grievances within 15 days. Penalties apply for delays exceeding 30 days."
)

rprint(Panel(
    f"[bold]MCP Server:[/bold] {insurance_mcp.name} v{insurance_mcp.version}\n\n"
    f"[bold]Tools registered:[/bold]\n" +
    "\n".join([f"  • {t['name']}: {t['description']}" for t in insurance_mcp.list_tools()]),
    title="[cyan]5.1 MCP Server — Insurance Domain[/cyan]",
    border_style="cyan"
))


╭─────────────────────────────────────── 5.1 MCP Server — Insurance Domain ───────────────────────────────────────╮
│ MCP Server: insurance-tools v1.0.0                                                                              │
│                                                                                                                 │
│ Tools registered:                                                                                               │
│   • lookup_policy: Look up an insurance policy by policy number                                                 │
│   • submit_grievance: Submit a grievance for a policy                                                           │
│   • check_winnability: Predict whether a grievance is likely to be resolved in claimant's favor                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [14]:
# ─── 5.2 MCP CLIENT AGENT ────────────────────────────────────────────────────
# Agent that dynamically discovers MCP tools and uses them via standard protocol.
# Key: tool discovery at runtime (not hardcoded).

class MCPAgentState(TypedDict):
    messages: Annotated[list, add_messages]
    mcp_server: Any             # ← reference to MCP server
    available_tools: List[Dict] # ← discovered at runtime
    tool_calls_made: List[Dict] # ← for evaluation
    final_answer: str


def mcp_discovery_node(state: MCPAgentState) -> dict:
    """Discover available tools from MCP server (tools/list)"""
    server: MCPServer = state["mcp_server"]
    tools = server.list_tools()
    
    tool_summary = "\n".join([f"  • {t['name']}: {t['description']}" for t in tools])
    
    return {
        "available_tools": tools,
        "messages": [AIMessage(content=f"Discovered {len(tools)} MCP tools:\n{tool_summary}")],
    }


# Scripted tool selection for demo
SCRIPTED_MCP_CALLS = [
    {"tool": "lookup_policy", "args": {"policy_id": "POL-2024-8821"}},
    {"tool": "submit_grievance", "args": {
        "policy_id": "POL-2024-8821", 
        "category": "claim_rejection",
        "description": "Claim rejected without adequate reason. Policy terms clearly cover the incident."
    }},
    {"tool": "check_winnability", "args": {
        "grievance_id": "PENDING",  # will be replaced
        "evidence_summary": "Strong documentation, clear policy coverage, IRDAI precedent favorable"
    }},
]


def mcp_planner_node(state: MCPAgentState) -> dict:
    """Decide which MCP tools to call and in what order"""
    call_plan = SCRIPTED_MCP_CALLS.copy()
    return {
        "messages": [AIMessage(content=f"Planned {len(call_plan)} MCP tool calls")],
        "tool_calls_made": call_plan,
    }


def mcp_executor_node(state: MCPAgentState) -> dict:
    """Execute all planned MCP tool calls"""
    server: MCPServer = state["mcp_server"]
    calls = state.get("tool_calls_made", [])
    
    results = []
    grievance_id = None
    
    for call in calls:
        tool_name = call["tool"]
        args = call["args"].copy()
        
        # Chain: use result from previous call
        if "grievance_id" in args and args["grievance_id"] == "PENDING" and grievance_id:
            args["grievance_id"] = grievance_id
        
        result = server.call_tool(tool_name, args)
        result_data = json.loads(result.content[0]["text"])
        
        # Extract grievance_id for chaining
        if "grievance_id" in result_data:
            grievance_id = result_data["grievance_id"]
        
        results.append({
            "tool": tool_name,
            "args": args,
            "result": result_data,
            "error": result.is_error,
        })
    
    summary_parts = []
    for r in results:
        summary_parts.append(f"[{r['tool']}]: {json.dumps(r['result'])[:100]}")
    
    final = "\n".join(summary_parts)
    
    return {
        "tool_calls_made": results,
        "final_answer": final,
        "messages": [AIMessage(content=f"Executed {len(results)} MCP calls:\n{final}")],
    }


def build_mcp_agent():
    builder = StateGraph(MCPAgentState)
    builder.add_node("discover", mcp_discovery_node)
    builder.add_node("plan", mcp_planner_node)
    builder.add_node("execute", mcp_executor_node)
    
    builder.add_edge(START, "discover")
    builder.add_edge("discover", "plan")
    builder.add_edge("plan", "execute")
    builder.add_edge("execute", END)
    
    return builder.compile()


mcp_agent = build_mcp_agent()
mcp_result = mcp_agent.invoke({
    "messages": [HumanMessage(content="Process grievance for policy POL-2024-8821 (claim rejection)")],
    "mcp_server": insurance_mcp,
    "available_tools": [],
    "tool_calls_made": [],
    "final_answer": "",
})

rprint(Panel("[bold]5.2 MCP Agent Execution[/bold]", border_style="cyan"))
for call in mcp_result["tool_calls_made"]:
    color = "green" if not call.get("error") else "red"
    rprint(f"  [bold]{call['tool']}[/bold]")
    rprint(f"    Args: {json.dumps(call['args'])[:80]}...")
    rprint(f"    [{color}]Result: {json.dumps(call['result'])[:100]}[/{color}]")
    rprint()

rprint(f"[bold]MCP Server call log:[/bold] {len(insurance_mcp.call_log)} calls recorded")


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 5.2 MCP Agent Execution                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

lookup_policy

Args: {"policy_id": "POL-2024-8821"}...

Result: {"policy_id": "POL-2024-8821", "holder": "John Doe", "type": "Term Life", "premium": 12500, "status"

submit_grievance

...

Result: {"grievance_id": "GR-CB4B74", "status": "submitted", "policy_id": "POL-2024-8821", "category": "clai

check_winnability

Args: {"grievance_id": "GR-CB4B74", "evidence_summary": "Strong documentation, clear p...

Result: {"grievance_id": "GR-CB4B74", "winnability_score": 0.5, "confidence": "Medium", "key_factors": ["doc

MCP Server call log: 3 calls recorded

In [15]:
# ─── 5.3 PARALLEL TOOL CALLING ────────────────────────────────────────────────
# Modern LLMs can issue multiple tool calls in a single turn.
# This is more efficient but requires careful evaluation of call ordering.

import asyncio

@tool
async def async_search(query: str) -> str:
    """Async web search mock"""
    await asyncio.sleep(0.05)  # simulate network latency
    return f"Search results for '{query}': Found 10 relevant documents."


@tool  
async def async_database_lookup(table: str, filter_key: str) -> str:
    """Async database lookup mock"""
    await asyncio.sleep(0.03)
    data = {"policies": "Found 150 policies", "grievances": "Found 23 open grievances", "claims": "Found 7 pending claims"}
    return data.get(table, f"Table '{table}' not found")


@tool
async def async_ml_prediction(model: str, input_data: str) -> str:
    """Async ML model inference mock"""
    await asyncio.sleep(0.08)
    score = round(random.uniform(0.6, 0.95), 3)
    return f"Model '{model}' prediction: score={score}, confidence=High"


PARALLEL_TOOLS = {
    "async_search": async_search,
    "async_database_lookup": async_database_lookup,
    "async_ml_prediction": async_ml_prediction,
}


class ParallelToolState(TypedDict):
    messages: Annotated[list, add_messages]
    parallel_calls: List[Dict]    # multiple tool calls at once
    results: Dict[str, str]
    total_latency_ms: float


async def parallel_tool_executor(state: ParallelToolState) -> dict:
    """Execute all tool calls concurrently (not sequentially)"""
    calls = state["parallel_calls"]
    start = time.time()
    
    # Create all coroutines
    async def run_one(call):
        tool_fn = PARALLEL_TOOLS.get(call["name"])
        if tool_fn:
            result = await tool_fn.ainvoke(call["args"])
            return call["name"], result
        return call["name"], f"Tool {call['name']} not found"
    
    # Run ALL in parallel via asyncio.gather
    pairs = await asyncio.gather(*[run_one(c) for c in calls])
    results = dict(pairs)
    
    total_ms = (time.time() - start) * 1000
    
    return {
        "results": results,
        "total_latency_ms": total_ms,
        "messages": [AIMessage(content=f"Parallel execution: {len(calls)} tools in {total_ms:.0f}ms")],
    }


# Demo: 3 tools in parallel
parallel_calls_demo = [
    {"name": "async_search", "args": {"query": "insurance grievance resolution India"}},
    {"name": "async_database_lookup", "args": {"table": "grievances", "filter_key": "open"}},
    {"name": "async_ml_prediction", "args": {"model": "winnability-v2", "input_data": "claim_rejection_policy_payout"}},
]


async def run_parallel_demo():
    state = {
        "messages": [HumanMessage(content="Run parallel analysis")],
        "parallel_calls": parallel_calls_demo,
        "results": {},
        "total_latency_ms": 0.0,
    }
    result = await parallel_tool_executor(state)
    return result

result_parallel = asyncio.run(run_parallel_demo())

# Compare vs sequential time estimate
seq_estimate = 50 + 30 + 80  # ms per tool
parallel_time = result_parallel["total_latency_ms"]
speedup = seq_estimate / parallel_time if parallel_time > 0 else 1.0

rprint(Panel(
    "[bold]Parallel vs Sequential Tool Execution:[/bold]\n\n" +
    "\n".join([f"  [green]✅ {k}:[/green] {v}" for k, v in result_parallel["results"].items()]) +
    f"\n\n[bold]Parallel time:[/bold]  {parallel_time:.0f}ms"
    f"\n[bold]Sequential est:[/bold] {seq_estimate}ms"
    f"\n[bold yellow]Speedup: {speedup:.1f}x faster[/bold yellow]",
    title="[cyan]5.3 Parallel Tool Calling[/cyan]",
    border_style="cyan"
))


RuntimeError: asyncio.run() cannot be called from a running event loop

---
## ⚙️ Section 6: Agentic Workflows

Workflows are higher-level patterns that compose agents:

| Pattern | Description | Use Case |
|---------|-------------|----------|
| **Sequential** | A → B → C (pipeline) | ETL, document processing |
| **Parallel (Fan-out/in)** | A → [B,C,D] → E | Multi-source aggregation |
| **Map-Reduce** | Split → [parallel map] → reduce | Batch processing |
| **Conditional** | Route based on content | Classification-first pipelines |
| **Human-in-the-Loop** | Pause for human approval | High-stakes decisions |
| **Long-Running** | Persist state, resume later | Days-long research tasks |


In [16]:
# ─── 6.1 FAN-OUT / FAN-IN WORKFLOW ───────────────────────────────────────────
# Send work to multiple branches in parallel, then aggregate results.
# LangGraph supports this via Send() API for dynamic parallel edges.

from langgraph.types import Send

class FanOutState(TypedDict):
    messages: Annotated[list, add_messages]
    documents: List[str]
    analyses: Annotated[List[Dict], operator.add]   # ← reducer: collects from all branches
    final_summary: str


class SingleDocState(TypedDict):
    document: str
    doc_id: int
    analysis: Dict


def analyze_single_doc(state: SingleDocState) -> Dict:
    """Analyze one document — runs in parallel across all docs"""
    doc = state["document"]
    doc_id = state["doc_id"]
    
    # Mock analysis
    words = doc.split()
    return {
        "analyses": [{
            "doc_id": doc_id,
            "word_count": len(words),
            "key_terms": [w for w in words if len(w) > 6][:3],
            "sentiment": random.choice(["positive", "neutral", "informative"]),
            "doc_preview": doc[:50] + "...",
        }]
    }


def fan_out_node(state: FanOutState) -> List[Send]:
    """Generate a Send for each document → parallel execution"""
    return [
        Send("analyze_doc", {"document": doc, "doc_id": i, "analysis": {}})
        for i, doc in enumerate(state["documents"])
    ]


def aggregator_node(state: FanOutState) -> dict:
    """Fan-in: aggregate all parallel results"""
    analyses = state.get("analyses", [])
    
    total_words = sum(a["word_count"] for a in analyses)
    all_terms = [t for a in analyses for t in a.get("key_terms", [])]
    sentiments = [a["sentiment"] for a in analyses]
    
    summary = (
        f"Analyzed {len(analyses)} documents. "
        f"Total words: {total_words}. "
        f"Top terms: {', '.join(set(all_terms)[:5])}. "
        f"Sentiments: {dict((s, sentiments.count(s)) for s in set(sentiments))}."
    )
    
    return {
        "final_summary": summary,
        "messages": [AIMessage(content=summary)],
    }


def build_fan_out_workflow():
    builder = StateGraph(FanOutState)
    builder.add_node("fan_out", fan_out_node)
    builder.add_node("analyze_doc", analyze_single_doc)
    builder.add_node("aggregate", aggregator_node)
    
    builder.add_edge(START, "fan_out")
    builder.add_conditional_edges("fan_out", lambda s: s, ["analyze_doc"])  # Send API
    builder.add_edge("analyze_doc", "aggregate")
    builder.add_edge("aggregate", END)
    
    return builder.compile()


# ── Fallback: sequential fan-out simulation ────────────────────────────────────
# (LangGraph Send API has version-specific nuances; this always works)
def fan_out_sequential_fallback(documents: List[str]) -> Dict:
    """Process documents 'in parallel' (sequential for demo, same result)"""
    analyses = []
    for i, doc in enumerate(documents):
        state = {"document": doc, "doc_id": i, "analysis": {}}
        result = analyze_single_doc(state)
        analyses.extend(result["analyses"])
    
    # Aggregate
    total_words = sum(a["word_count"] for a in analyses)
    all_terms = [t for a in analyses for t in a.get("key_terms", [])]
    sentiments = [a["sentiment"] for a in analyses]
    
    return {
        "analyses": analyses,
        "final_summary": (
            f"Analyzed {len(analyses)} docs. Total words: {total_words}. "
            f"Terms: {', '.join(set(all_terms)[:5])}. "
            f"Sentiments: {dict((s, sentiments.count(s)) for s in set(sentiments))}."
        )
    }


test_docs = [
    "Agent evaluation frameworks provide systematic methods for measuring LLM agent performance.",
    "DeepEval supports hallucination detection, answer relevancy, and contextual precision metrics.",
    "RAGAS evaluates RAG pipelines with faithfulness, context recall, and answer correctness.",
    "LangGraph enables stateful multi-agent workflows with checkpointing and persistence.",
    "MCP standardizes how AI agents connect to tools and data sources via JSON-RPC protocol.",
]

fan_result = fan_out_sequential_fallback(test_docs)

table = Table(title="Fan-Out Analysis Results", header_style="bold cyan")
table.add_column("Doc ID"); table.add_column("Words"); table.add_column("Key Terms"); table.add_column("Sentiment")
for a in fan_result["analyses"]:
    table.add_row(
        str(a["doc_id"]), str(a["word_count"]),
        ", ".join(a["key_terms"]),
        f"[yellow]{a['sentiment']}[/yellow]"
    )
console.print(table)
rprint(f"\n[bold green]Summary:[/bold green] {fan_result['final_summary']}")


TypeError: 'set' object is not subscriptable

In [17]:
# ─── 6.2 HUMAN-IN-THE-LOOP WORKFLOW ──────────────────────────────────────────
# Critical for high-stakes decisions: agent pauses, human reviews/approves.
# LangGraph uses interrupt() to pause and MemorySaver to resume from checkpoint.

from langgraph.types import interrupt

class HITLState(TypedDict):
    messages: Annotated[list, add_messages]
    proposed_action: str
    human_decision: str    # "approve" | "reject" | "modify"
    modification: str
    final_action: str
    risk_level: str        # "low" | "medium" | "high"


def agent_propose_node(state: HITLState) -> dict:
    """Agent proposes an action that requires human review"""
    # Simulate agent deciding a high-stakes action
    proposal = "PROPOSED: Submit regulatory complaint to IRDAI against insurer for claim delay of 45+ days. Estimated claim value: ₹2.5L."
    risk = "high"
    
    return {
        "proposed_action": proposal,
        "risk_level": risk,
        "messages": [AIMessage(content=f"[Agent Proposal - Risk: {risk}]:\n{proposal}\n\nAwaiting human approval...")],
    }


def human_review_node(state: HITLState) -> dict:
    """
    HUMAN-IN-THE-LOOP: agent PAUSES here.
    In production: interrupt() suspends graph, human approves via UI/API.
    We simulate with a scripted response.
    """
    proposed = state.get("proposed_action", "")
    
    # In production: value = interrupt({"proposed_action": proposed, "risk_level": state["risk_level"]})
    # For demo: simulate human saying "approve with modification"
    simulated_human_input = {
        "decision": "modify",
        "modification": "Include additional documentation requirement: attach all previous rejection letters",
        "approved_by": "Harsh V. Singh",
        "timestamp": datetime.now().isoformat(),
    }
    
    return {
        "human_decision": simulated_human_input["decision"],
        "modification": simulated_human_input.get("modification", ""),
        "messages": [
            HumanMessage(content=f"[Human Review by {simulated_human_input['approved_by']}]: "
                                  f"Decision: {simulated_human_input['decision']}. "
                                  f"Note: {simulated_human_input.get('modification', 'None')}")
        ],
    }


def execute_with_human_input_node(state: HITLState) -> dict:
    """Execute the action incorporating human feedback"""
    base = state.get("proposed_action", "")
    modification = state.get("modification", "")
    decision = state.get("human_decision", "reject")
    
    if decision == "approve":
        final = f"EXECUTED: {base}"
    elif decision == "modify":
        final = f"EXECUTED (modified): {base}. MODIFICATION: {modification}"
    else:
        final = f"REJECTED: Action not taken. Reason: Human reviewer rejected the proposal."
    
    return {
        "final_action": final,
        "messages": [AIMessage(content=f"Final outcome: {final}")],
    }


def hitl_router(state: HITLState) -> Literal["execute", "reject"]:
    return "reject" if state.get("human_decision") == "reject" else "execute"


def build_hitl_workflow():
    builder = StateGraph(HITLState)
    builder.add_node("propose", agent_propose_node)
    builder.add_node("human_review", human_review_node)
    builder.add_node("execute", execute_with_human_input_node)
    
    builder.add_edge(START, "propose")
    builder.add_edge("propose", "human_review")
    builder.add_conditional_edges("human_review", hitl_router, {
        "execute": "execute",
        "reject": END,
    })
    builder.add_edge("execute", END)
    
    memory = MemorySaver()
    return builder.compile(checkpointer=memory)


hitl_workflow = build_hitl_workflow()
hitl_config = {"configurable": {"thread_id": "hitl-demo-001"}}

hitl_result = hitl_workflow.invoke({
    "messages": [HumanMessage(content="Process grievance for 45-day delayed claim")],
    "proposed_action": "", "human_decision": "", "modification": "",
    "final_action": "", "risk_level": "",
}, config=hitl_config)

rprint(Panel(
    f"[bold]Risk Level:[/bold] [red]{hitl_result['risk_level']}[/red]\n\n"
    f"[bold]Proposed:[/bold]\n  {hitl_result['proposed_action']}\n\n"
    f"[bold]Human Decision:[/bold] [yellow]{hitl_result['human_decision']}[/yellow]\n"
    f"[bold]Modification:[/bold] {hitl_result.get('modification', 'N/A')}\n\n"
    f"[bold green]Final Action:[/bold green]\n  {hitl_result['final_action']}",
    title="[cyan]6.2 Human-in-the-Loop Workflow[/cyan]",
    border_style="cyan"
))


╭──────────────────────────────────────── 6.2 Human-in-the-Loop Workflow ─────────────────────────────────────────╮
│ Risk Level: high                                                                                                │
│                                                                                                                 │
│ Proposed:                                                                                                       │
│   PROPOSED: Submit regulatory complaint to IRDAI against insurer for claim delay of 45+ days. Estimated claim   │
│ value: ₹2.5L.                                                                                                   │
│                                                                                                                 │
│ Human Decision: modify                                                                                          │
│ Modification: Include additional documentation requirement: attach all previous rejection letters               │
│                                                                                                                 │
│ Final Action:                                                                                                   │
│   EXECUTED (modified): PROPOSED: Submit regulatory complaint to IRDAI against insurer for claim delay of 45+    │
│ days. Estimated claim value: ₹2.5L.. MODIFICATION: Include additional documentation requirement: attach all     │
│ previous rejection letters                                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## 📐 Section 7: Agent Evaluation — Foundations

Before using any framework, you must understand the *taxonomy* of agent evaluation.

### 7.1 The Evaluation Pyramid

```
                    ┌─────────────────────┐
                    │   SYSTEM EVAL       │  ← End-to-end business KPIs
                    │ (task success rate, │
                    │  user satisfaction) │
                ┌───┴─────────────────────┴───┐
                │    WORKFLOW EVAL            │  ← Multi-step correctness,
                │  (trajectory, efficiency)   │     safety, latency
            ┌───┴─────────────────────────────┴───┐
            │         COMPONENT EVAL              │  ← Per-LLM-call quality
            │  (faithfulness, relevancy, tone)    │
        ┌───┴─────────────────────────────────────┴───┐
        │              UNIT EVAL                      │  ← Tool accuracy, 
        │    (tool selection, format correctness)     │     schema validation
        └─────────────────────────────────────────────┘
```

### 7.2 Evaluation Metric Taxonomy

| Dimension | Metrics | Framework |
|-----------|---------|-----------|
| **Correctness** | Exact match, semantic similarity, F1 | Custom |
| **Faithfulness** | Grounded in context? | RAGAS, DeepEval |
| **Relevancy** | Answer relevant to question? | RAGAS, DeepEval |
| **Tool Use** | Right tools, right args, right order | Custom |
| **Trajectory** | Did agent take optimal path? | Custom |
| **Safety** | No harmful/toxic outputs | DeepEval, Custom |
| **Latency** | Response time per SLA | Custom |
| **Multi-turn** | Context retention across turns | Custom |
| **Groundedness** | No hallucinations | DeepEval |
| **Cost** | Token usage per task | Custom |


In [18]:
# ─── 7.1 TRACE LOGGING INFRASTRUCTURE ────────────────────────────────────────
# Production evaluation requires complete traces: every LLM call, tool call,
# state transition, and timing event must be captured.

@dataclass
class Span:
    """A single timed event in an agent trace"""
    span_id: str
    parent_id: Optional[str]
    span_type: str   # "llm_call" | "tool_call" | "node_enter" | "node_exit"
    name: str
    input: Any
    output: Any
    start_time: float
    end_time: float
    metadata: Dict = field(default_factory=dict)
    
    @property
    def duration_ms(self) -> float:
        return (self.end_time - self.start_time) * 1000
    
    def to_dict(self) -> Dict:
        return {
            "span_id": self.span_id,
            "parent_id": self.parent_id,
            "type": self.span_type,
            "name": self.name,
            "input_summary": str(self.input)[:100],
            "output_summary": str(self.output)[:100],
            "duration_ms": round(self.duration_ms, 2),
            "metadata": self.metadata,
        }


@dataclass
class AgentTrace:
    """Complete trace of an agent run"""
    trace_id: str
    task: str
    spans: List[Span] = field(default_factory=list)
    start_time: float = field(default_factory=time.time)
    end_time: Optional[float] = None
    error: Optional[str] = None
    
    def add_span(self, span: Span):
        self.spans.append(span)
    
    @property
    def total_duration_ms(self) -> float:
        return (self.end_time - self.start_time) * 1000 if self.end_time else 0.0
    
    @property
    def llm_calls(self) -> List[Span]:
        return [s for s in self.spans if s.span_type == "llm_call"]
    
    @property
    def tool_calls(self) -> List[Span]:
        return [s for s in self.spans if s.span_type == "tool_call"]
    
    @property
    def total_llm_tokens(self) -> int:
        return sum(s.metadata.get("tokens", 0) for s in self.llm_calls)
    
    def summary(self) -> Dict:
        return {
            "trace_id": self.trace_id,
            "task": self.task[:50],
            "total_ms": round(self.total_duration_ms, 1),
            "llm_calls": len(self.llm_calls),
            "tool_calls": len(self.tool_calls),
            "total_spans": len(self.spans),
            "status": "error" if self.error else "success",
        }


class TraceCollector:
    """
    Collects traces from agent runs. Attach to LangGraph via callbacks.
    In production: export to LangSmith, Weights & Biases, Arize, etc.
    """
    def __init__(self):
        self.traces: List[AgentTrace] = []
        self._active_trace: Optional[AgentTrace] = None
    
    def start_trace(self, task: str) -> AgentTrace:
        trace = AgentTrace(
            trace_id=f"trace-{uuid.uuid4().hex[:8]}",
            task=task,
        )
        self._active_trace = trace
        self.traces.append(trace)
        return trace
    
    def end_trace(self, error: str = None):
        if self._active_trace:
            self._active_trace.end_time = time.time()
            self._active_trace.error = error
    
    def log_llm_call(self, name: str, prompt: str, response: str, tokens: int = 80):
        if not self._active_trace:
            return
        start = time.time()
        span = Span(
            span_id=f"span-{uuid.uuid4().hex[:6]}",
            parent_id=self._active_trace.trace_id,
            span_type="llm_call",
            name=name,
            input=prompt,
            output=response,
            start_time=start,
            end_time=start + random.uniform(0.05, 0.3),
            metadata={"tokens": tokens, "model": "mock-llm"},
        )
        self._active_trace.add_span(span)
    
    def log_tool_call(self, tool_name: str, args: Dict, result: str):
        if not self._active_trace:
            return
        start = time.time()
        span = Span(
            span_id=f"span-{uuid.uuid4().hex[:6]}",
            parent_id=self._active_trace.trace_id,
            span_type="tool_call",
            name=tool_name,
            input=args,
            output=result,
            start_time=start,
            end_time=start + random.uniform(0.01, 0.1),
            metadata={"tool": tool_name},
        )
        self._active_trace.add_span(span)
    
    def get_dataframe(self) -> pd.DataFrame:
        rows = [t.summary() for t in self.traces]
        return pd.DataFrame(rows) if rows else pd.DataFrame()


# ── Simulate traces from agent runs ──────────────────────────────────────────
collector = TraceCollector()

# Simulate 5 agent runs
SAMPLE_TASKS = [
    ("Research agent evaluation best practices", True),
    ("Process insurance grievance POL-001", True),
    ("Calculate risk score for claim", True),
    ("Generate compliance report", False),  # ← will be an error
    ("Multi-document summarization", True),
]

for task, should_succeed in SAMPLE_TASKS:
    trace = collector.start_trace(task)
    
    # Add LLM calls
    collector.log_llm_call("planner", f"Plan: {task}", "Step 1: Research. Step 2: Execute.", tokens=120)
    collector.log_llm_call("executor", "Execute step 1", f"Result for {task[:30]}", tokens=80)
    
    # Add tool calls (random)
    num_tools = random.randint(1, 3)
    for j in range(num_tools):
        collector.log_tool_call(
            random.choice(["calculator", "web_search_mock", "lookup_policy"]),
            {"query": task[:30]},
            f"Tool result {j+1}"
        )
    
    # Add final LLM call
    collector.log_llm_call("synthesizer", "Synthesize results", f"Final answer for: {task[:30]}", tokens=150)
    
    error = None if should_succeed else "timeout: exceeded 30s SLA"
    collector.end_trace(error=error)

# Show trace summary
df = collector.get_dataframe()
rprint(Panel("[bold]7.1 Trace Collection Summary[/bold]", border_style="green"))
console.print(df.to_string(index=False))
rprint(f"\n[green]✅ {len(collector.traces)} traces collected[/green]")
rprint(f"[dim]Errors: {df['status'].value_counts().get('error', 0)}, "
      f"Success: {df['status'].value_counts().get('success', 0)}[/dim]")


c:\Program Files\Python313\Lib\collections\__init__.py:450: RuntimeWarning: coroutine 'run_parallel_demo' was never awaited
  @classmethod


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 7.1 Trace Collection Summary                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

trace_id                                     task  total_ms  llm_calls  tool_calls  total_spans  status
trace-d2c9bc0e Research agent evaluation best practices       0.1          3           1            4 success
trace-ff1a6951      Process insurance grievance POL-001       0.0          3           1            4 success
trace-ca34c32b           Calculate risk score for claim       0.0          3           1            4 success
trace-cf61c9a2               Generate compliance report       0.0          3           3            6   error
trace-9cda8c34             Multi-document summarization       0.1          3           3            6 success

✅ 5 traces collected

Errors: 1, Success: 4

In [19]:
# ─── 7.2 CUSTOM METRICS LIBRARY ──────────────────────────────────────────────
# Build reusable evaluator classes. These become your evaluation primitives.

class BaseMetric:
    """Abstract base for all evaluation metrics"""
    name: str = "base_metric"
    threshold: float = 0.7
    
    def score(self, **kwargs) -> float:
        raise NotImplementedError
    
    def evaluate(self, **kwargs) -> EvalResult:
        s = self.score(**kwargs)
        return EvalResult(
            metric=self.name, score=s,
            passed=s >= self.threshold,
            reason=self._reason(s, **kwargs),
        )
    
    def _reason(self, score: float, **kwargs) -> str:
        return f"Score: {score:.3f}"


class SemanticSimilarityMetric(BaseMetric):
    """
    Semantic similarity using token overlap (Jaccard) as a proxy.
    Production: replace with sentence-transformers cosine similarity.
    
    pip install sentence-transformers
    from sentence_transformers import SentenceTransformer, util
    model = SentenceTransformer('all-MiniLM-L6-v2')
    similarity = util.cos_sim(model.encode(a), model.encode(b)).item()
    """
    name = "semantic_similarity"
    
    def score(self, actual: str, expected: str, **kwargs) -> float:
        a_words = set(actual.lower().split())
        e_words = set(expected.lower().split())
        if not a_words or not e_words:
            return 0.0
        jaccard = len(a_words & e_words) / len(a_words | e_words)
        return round(jaccard, 3)
    
    def _reason(self, score, actual="", expected="", **kwargs):
        return f"Token overlap: {score:.3f} (Jaccard). For production, use sentence-transformers."


class HallucinationMetric(BaseMetric):
    """
    Detects hallucination: claims in output not supported by context.
    Production: use DeepEval HallucinationMetric with LLM judge.
    """
    name = "hallucination"
    threshold = 0.5  # lower score = more hallucination
    
    def score(self, actual: str, context: str, **kwargs) -> float:
        """Estimate: what fraction of unique output words appear in context?"""
        # Filter to content words (>4 chars)
        actual_words = set(w for w in actual.lower().split() if len(w) > 4)
        context_words = set(w for w in context.lower().split() if len(w) > 4)
        
        if not actual_words:
            return 1.0  # empty output = no hallucination
        
        grounded = len(actual_words & context_words) / len(actual_words)
        return round(grounded, 3)
    
    def _reason(self, score, actual="", context="", **kwargs):
        if score >= 0.7:
            return f"Mostly grounded ({score:.3f}). Low hallucination risk."
        elif score >= 0.5:
            return f"Partially grounded ({score:.3f}). Some hallucination risk."
        else:
            return f"Poorly grounded ({score:.3f}). High hallucination risk!"


class AnswerRelevancyMetric(BaseMetric):
    """
    Does the answer address the question?
    Production: LLM judge or NLI model.
    """
    name = "answer_relevancy"
    
    def score(self, question: str, answer: str, **kwargs) -> float:
        q_words = set(q for q in question.lower().split() if len(q) > 3)
        a_words = set(a for a in answer.lower().split() if len(a) > 3)
        
        if not q_words:
            return 1.0
        
        coverage = len(q_words & a_words) / len(q_words)
        length_penalty = min(1.0, len(answer.split()) / 20)  # too short = penalty
        return round(min(1.0, coverage * 1.5) * length_penalty, 3)


class TrajectoryAccuracyMetric(BaseMetric):
    """
    Did the agent follow the expected node sequence?
    Measures: edit distance between expected and actual trajectory.
    """
    name = "trajectory_accuracy"
    
    def _edit_distance(self, a: List[str], b: List[str]) -> int:
        """Levenshtein distance between two sequences"""
        if not a: return len(b)
        if not b: return len(a)
        m, n = len(a), len(b)
        dp = [[0]*(n+1) for _ in range(m+1)]
        for i in range(m+1): dp[i][0] = i
        for j in range(n+1): dp[0][j] = j
        for i in range(1,m+1):
            for j in range(1,n+1):
                if a[i-1] == b[j-1]:
                    dp[i][j] = dp[i-1][j-1]
                else:
                    dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
        return dp[m][n]
    
    def score(self, expected_trajectory: List[str], actual_trajectory: List[str], **kwargs) -> float:
        dist = self._edit_distance(expected_trajectory, actual_trajectory)
        max_len = max(len(expected_trajectory), len(actual_trajectory), 1)
        return round(1.0 - dist / max_len, 3)
    
    def _reason(self, score, expected_trajectory=None, actual_trajectory=None, **kwargs):
        return (f"Trajectory similarity: {score:.3f}. "
                f"Expected: {expected_trajectory} | Actual: {actual_trajectory}")


class SafetyMetric(BaseMetric):
    """
    Basic safety check: looks for harmful/sensitive patterns.
    Production: use DeepEval BiasMetric + ToxicityMetric + GuardrailsAI.
    """
    name = "safety"
    threshold = 0.9  # very high threshold for safety
    
    HARMFUL_PATTERNS = [
        r"\b(kill|murder|attack|bomb|poison|hack|steal|fraud)\b",
        r"\b(social security|credit card number|password|secret key)\b",
        r"\b(illegal|criminal|unlawful)\b(?!.*defense|academic|research)",
    ]
    
    def score(self, text: str, **kwargs) -> float:
        violations = 0
        for pattern in self.HARMFUL_PATTERNS:
            if re.search(pattern, text.lower()):
                violations += 1
        return round(max(0.0, 1.0 - violations * 0.5), 3)


# ── Test the metrics ──────────────────────────────────────────────────────────
metrics = [
    SemanticSimilarityMetric(),
    HallucinationMetric(),
    AnswerRelevancyMetric(),
    TrajectoryAccuracyMetric(),
    SafetyMetric(),
]

test_inputs = {
    "actual": "RAGAS evaluates RAG pipelines using faithfulness and answer relevancy metrics for LLM applications.",
    "expected": "RAGAS is a framework for evaluating retrieval augmented generation with faithfulness and relevancy scores.",
    "context": "RAGAS (Retrieval Augmented Generation Assessment) evaluates RAG pipelines on faithfulness, answer relevancy, context precision, and context recall metrics.",
    "question": "What does RAGAS evaluate?",
    "answer": "RAGAS evaluates RAG pipelines using faithfulness, answer relevancy, context precision metrics.",
    "expected_trajectory": ["retrieve", "grade_docs", "generate", "grade_answer"],
    "actual_trajectory": ["retrieve", "generate", "grade_answer"],
    "text": "RAGAS is a helpful evaluation framework for building better AI systems.",
}

table = Table(title="Custom Metrics Evaluation", header_style="bold magenta")
table.add_column("Metric"); table.add_column("Score"); table.add_column("Pass?"); table.add_column("Reason")

for metric in metrics:
    result = metric.evaluate(**test_inputs)
    color = "green" if result.passed else "red"
    table.add_row(
        metric.name,
        f"[{color}]{result.score:.3f}[/{color}]",
        f"[{color}]{'✅' if result.passed else '❌'}[/{color}]",
        result.reason[:70]
    )

console.print(table)


                                           Custom Metrics Evaluation                                            
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric              ┃ Score ┃ Pass? ┃ Reason                                                                 ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ semantic_similarity │ 0.227 │ ❌    │ Token overlap: 0.227 (Jaccard). For production, use sentence-transform │
│ hallucination       │ 0.444 │ ❌    │ Poorly grounded (0.444). High hallucination risk!                      │
│ answer_relevancy    │ 0.206 │ ❌    │ Score: 0.206                                                           │
│ trajectory_accuracy │ 0.750 │ ✅    │ Trajectory similarity: 0.750. Expected: ['retrieve', 'grade_docs', 'ge │
│ safety              │ 1.000 │ ✅    │ Score: 1.000                                                           │
└─────────────────────┴───────┴───────┴────────────────────────────────────────────────────────────────────────┘

---
## 🧪 Section 8: Evaluation Frameworks — DeepEval & RAGAS

Now we integrate industry-standard open-source frameworks.

### Framework Comparison

| Aspect | DeepEval | RAGAS | Custom |
|--------|----------|-------|--------|
| **Best for** | LLM unit testing | RAG pipelines | Domain-specific |
| **Requires LLM judge** | Yes (configurable) | Yes (configurable) | Optional |
| **Key metrics** | Hallucination, Relevancy, Contextual Precision, GEval | Faithfulness, Context Recall, Answer Correctness | Anything |
| **CI/CD** | ✅ Pytest integration | ⚠️ Manual | ✅ |
| **Open source** | ✅ | ✅ | ✅ |
| **Local LLM support** | ✅ | ✅ | ✅ |

> **Key insight:** Use DeepEval for *component tests* of individual agent nodes; use RAGAS for *end-to-end RAG pipeline* evaluation; use custom metrics for *domain-specific* requirements.


In [20]:
# ─── 8.1 DEEPEVAL FRAMEWORK ──────────────────────────────────────────────────
# DeepEval provides unit-test-style evaluation for LLM outputs.
# Key concept: LLMTestCase → metrics → evaluate()
# 
# DeepEval works best with a real LLM judge (GPT-4, Claude, etc.).
# We demonstrate the pattern using its pure Python metrics that don't need LLM.

try:
    from deepeval.test_case import LLMTestCase, LLMTestCaseParams
    from deepeval.metrics import (
        AnswerRelevancyMetric as DEAnswerRelevancy,
        FaithfulnessMetric as DEFaithfulness,
        ContextualPrecisionMetric,
        ContextualRecallMetric,
    )
    from deepeval import evaluate as deepeval_evaluate
    DEEPEVAL_AVAILABLE = True
    rprint("[green]✅ DeepEval imported successfully[/green]")
except ImportError as e:
    DEEPEVAL_AVAILABLE = False
    rprint(f"[yellow]⚠️  DeepEval import partial: {e}[/yellow]")
    rprint("[dim]DeepEval metrics requiring LLM judge will be shown as patterns only[/dim]")


# ── DeepEval Test Cases ───────────────────────────────────────────────────────
# An LLMTestCase captures: input, actual_output, expected_output, context
# This is the fundamental unit of evaluation in DeepEval.

deepeval_test_cases_raw = [
    {
        "input": "What is RAGAS used for?",
        "actual_output": "RAGAS is used for evaluating RAG (Retrieval Augmented Generation) pipelines by measuring faithfulness, answer relevancy, and context precision.",
        "expected_output": "RAGAS evaluates RAG systems for faithfulness and answer quality.",
        "retrieval_context": [
            "RAGAS (Retrieval Augmented Generation Assessment) evaluates RAG pipelines on faithfulness, answer relevancy, context precision, and context recall metrics.",
            "RAGAS was created to provide standardized evaluation for RAG-based LLM applications.",
        ],
        "context": ["RAGAS evaluates RAG pipelines on faithfulness, answer relevancy, context precision and context recall."],
    },
    {
        "input": "How does LangGraph handle memory?",
        "actual_output": "LangGraph handles memory through MemorySaver checkpointers that persist state across conversation turns using thread IDs.",
        "expected_output": "LangGraph uses MemorySaver and checkpointers for persistent state management.",
        "retrieval_context": [
            "LangGraph enables persistent memory via MemorySaver checkpointers, storing state per thread_id.",
        ],
        "context": ["LangGraph enables persistent memory via MemorySaver checkpointers."],
    },
    {
        "input": "What are the main insurance grievance categories?",
        "actual_output": "The main categories include claim rejection, payment delays, mis-selling of policies, and policy servicing issues. Each has different resolution timelines under IRDAI guidelines.",
        "expected_output": "Grievance categories: claim rejection, delay, mis-selling, policy service.",
        "retrieval_context": [
            "IRDAI categories: claim rejection, delay in claim processing, mis-selling, policy servicing.",
        ],
        "context": ["IRDAI grievance categories include claim rejection, delay, mis-selling, and policy servicing."],
    },
]

# ── Implement DeepEval pattern using our own LLM-free evaluators ───────────────
# These are the SAME metrics DeepEval provides, reimplemented without LLM judge
# (so they run without API keys). The full DeepEval versions use LLM judges.

class DeepEvalStyleFaithfulness:
    """
    Pattern: DeepEval FaithfulnessMetric
    Real usage:
        metric = FaithfulnessMetric(threshold=0.7, model="gpt-4o")
        test_case = LLMTestCase(input=..., actual_output=..., retrieval_context=[...])
        metric.measure(test_case)
        print(metric.score, metric.reason)
    """
    name = "faithfulness"
    threshold = 0.7
    
    def measure(self, input: str, actual_output: str, retrieval_context: List[str]) -> Dict:
        context_text = " ".join(retrieval_context).lower()
        output_words = [w for w in actual_output.lower().split() if len(w) > 4]
        
        if not output_words:
            return {"score": 1.0, "passed": True, "reason": "Empty output"}
        
        grounded = sum(1 for w in output_words if w in context_text) / len(output_words)
        
        return {
            "score": round(grounded, 3),
            "passed": grounded >= self.threshold,
            "reason": f"{'High' if grounded >= 0.7 else 'Low'} faithfulness: "
                      f"{grounded:.1%} of output words grounded in context.",
            "metric": self.name,
        }


class DeepEvalStyleAnswerRelevancy:
    """Pattern: DeepEval AnswerRelevancyMetric"""
    name = "answer_relevancy"
    threshold = 0.7
    
    def measure(self, input: str, actual_output: str, **kwargs) -> Dict:
        q_content = set(w for w in input.lower().split() if len(w) > 3)
        a_content = set(w for w in actual_output.lower().split() if len(w) > 3)
        
        if not q_content:
            return {"score": 1.0, "passed": True, "reason": "No question keywords"}
        
        coverage = len(q_content & a_content) / len(q_content)
        # Boost for longer, detailed answers
        detail_bonus = min(0.3, len(actual_output.split()) / 100)
        score = min(1.0, coverage + detail_bonus)
        
        return {
            "score": round(score, 3),
            "passed": score >= self.threshold,
            "reason": f"Answer covers {coverage:.0%} of question keywords. "
                      f"Detail bonus: {detail_bonus:.2f}",
            "metric": self.name,
        }


class DeepEvalStyleContextualPrecision:
    """Pattern: DeepEval ContextualPrecisionMetric"""
    name = "contextual_precision"
    threshold = 0.7
    
    def measure(self, input: str, expected_output: str, retrieval_context: List[str]) -> Dict:
        q_words = set(w.lower() for w in input.split() if len(w) > 3)
        relevant_docs = 0
        
        for ctx in retrieval_context:
            ctx_words = set(w.lower() for w in ctx.split() if len(w) > 3)
            overlap = len(q_words & ctx_words) / len(q_words) if q_words else 0
            if overlap >= 0.3:
                relevant_docs += 1
        
        score = relevant_docs / len(retrieval_context) if retrieval_context else 0.0
        
        return {
            "score": round(score, 3),
            "passed": score >= self.threshold,
            "reason": f"{relevant_docs}/{len(retrieval_context)} retrieved docs are relevant.",
            "metric": self.name,
        }


# ── Run DeepEval-style evaluation ─────────────────────────────────────────────
de_metrics = [
    DeepEvalStyleFaithfulness(),
    DeepEvalStyleAnswerRelevancy(),
    DeepEvalStyleContextualPrecision(),
]

rprint(Panel("[bold]8.1 DeepEval-Style Evaluation[/bold]", border_style="cyan"))

all_de_results = []
for i, tc in enumerate(deepeval_test_cases_raw):
    rprint(f"\n[bold]Test Case {i+1}:[/bold] {tc['input'][:60]}...")
    tc_results = []
    
    for metric in de_metrics:
        result = metric.measure(
            input=tc["input"],
            actual_output=tc["actual_output"],
            expected_output=tc.get("expected_output", ""),
            retrieval_context=tc.get("retrieval_context", []),
        )
        tc_results.append(result)
        color = "green" if result["passed"] else "red"
        rprint(f"  [{color}]{result['metric']:28s} {result['score']:.3f}  {'✅' if result['passed'] else '❌'}[/{color}]")
        rprint(f"  [dim]  → {result['reason'][:80]}[/dim]")
    
    all_de_results.append({"test_case": tc["input"][:40], "results": tc_results})

rprint("\n[bold]📌 DeepEval Real Usage (requires LLM API key):[/bold]")
rprint("""[dim]
  from deepeval.test_case import LLMTestCase
  from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
  from deepeval import evaluate
  
  # Set API key (or use local Ollama):
  # deepeval.login_with_confident_api_key("your-key")
  # OR: export OPENAI_API_KEY="sk-..."
  
  test_case = LLMTestCase(
      input="What is RAGAS?",
      actual_output="RAGAS evaluates RAG pipelines...",
      expected_output="RAGAS measures faithfulness and relevancy.",
      retrieval_context=["RAGAS evaluates RAG systems..."]
  )
  
  metrics = [
      FaithfulnessMetric(threshold=0.7),
      AnswerRelevancyMetric(threshold=0.7),
  ]
  
  # Run evaluation
  evaluate(test_cases=[test_case], metrics=metrics)
[/dim]""")


C:\Users\pc\AppData\Local\Temp\ipykernel_44016\4061207432.py:9: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


✅ DeepEval imported successfully

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 8.1 DeepEval-Style Evaluation                                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Test Case 1: What is RAGAS used for?...

TypeError: DeepEvalStyleFaithfulness.measure() got an unexpected keyword argument 'expected_output'

In [21]:
# ─── 8.2 RAGAS FRAMEWORK ──────────────────────────────────────────────────────
# RAGAS (Retrieval Augmented Generation Assessment) specializes in RAG evaluation.
# Core metrics: faithfulness, answer_relevancy, context_precision, context_recall
#
# RAGAS metrics work on a Dataset with specific columns:
# question, answer, contexts, ground_truth

# ── RAGAS Core Metric Implementations (pure Python) ───────────────────────────
# These mirror RAGAS internal logic without requiring an LLM judge.

class RAGASFaithfulness:
    """
    RAGAS Faithfulness: measures if answer is grounded in context.
    
    Algorithm:
    1. Extract claims from the answer
    2. For each claim, check if it can be inferred from context
    3. Score = supported_claims / total_claims
    
    Real RAGAS: uses LLM to extract claims and verify inference.
    Our version: uses sentence overlap as a proxy.
    """
    name = "ragas_faithfulness"
    
    def __init__(self):
        self.threshold = 0.7
    
    def _extract_claims(self, text: str) -> List[str]:
        """Split text into atomic claims (sentences here as proxy)"""
        sentences = [s.strip() for s in text.split(".") if len(s.strip()) > 20]
        return sentences
    
    def _is_supported(self, claim: str, context: str) -> bool:
        """Check if claim words are present in context"""
        claim_words = set(w.lower() for w in claim.split() if len(w) > 4)
        context_words = set(w.lower() for w in context.split() if len(w) > 4)
        if not claim_words:
            return True
        overlap = len(claim_words & context_words) / len(claim_words)
        return overlap >= 0.4
    
    def score(self, answer: str, contexts: List[str]) -> float:
        claims = self._extract_claims(answer)
        if not claims:
            return 1.0
        context_text = " ".join(contexts)
        supported = sum(1 for c in claims if self._is_supported(c, context_text))
        return round(supported / len(claims), 3)


class RAGASAnswerRelevancy:
    """
    RAGAS Answer Relevancy: measures if answer addresses the question.
    
    Real RAGAS algorithm:
    1. Generate n questions from the answer using LLM
    2. Compute cosine similarity between generated questions and original question
    3. Score = mean similarity
    """
    name = "ragas_answer_relevancy"
    
    def score(self, question: str, answer: str) -> float:
        q_words = set(w.lower() for w in question.split() if len(w) > 3)
        a_words = set(w.lower() for w in answer.split() if len(w) > 3)
        
        if not q_words:
            return 1.0
        
        # Bidirectional overlap
        precision = len(q_words & a_words) / len(a_words) if a_words else 0.0
        recall = len(q_words & a_words) / len(q_words)
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        
        # Length bonus (longer answers tend to be more relevant if on topic)
        length_factor = min(1.2, max(0.8, len(answer.split()) / 30))
        return round(min(1.0, f1 * length_factor), 3)


class RAGASContextPrecision:
    """
    RAGAS Context Precision: measures if retrieved contexts are relevant to the question.
    Score = (# relevant contexts) / (# total contexts retrieved)
    """
    name = "ragas_context_precision"
    
    def score(self, question: str, contexts: List[str], ground_truth: str) -> float:
        q_words = set(w.lower() for w in (question + " " + ground_truth).split() if len(w) > 3)
        
        relevant = 0
        for ctx in contexts:
            ctx_words = set(w.lower() for w in ctx.split() if len(w) > 3)
            overlap = len(q_words & ctx_words) / len(q_words) if q_words else 0
            if overlap >= 0.25:
                relevant += 1
        
        return round(relevant / len(contexts), 3) if contexts else 0.0


class RAGASContextRecall:
    """
    RAGAS Context Recall: measures if contexts cover the ground truth.
    Score = (ground_truth claims supported by context) / (total ground truth claims)
    """
    name = "ragas_context_recall"
    
    def score(self, ground_truth: str, contexts: List[str]) -> float:
        gt_sentences = [s.strip() for s in ground_truth.split(".") if len(s.strip()) > 10]
        context_text = " ".join(contexts).lower()
        
        if not gt_sentences:
            return 1.0
        
        supported = 0
        for sent in gt_sentences:
            sent_words = set(w.lower() for w in sent.split() if len(w) > 4)
            ctx_words = set(w.lower() for w in context_text.split() if len(w) > 4)
            if len(sent_words) > 0 and len(sent_words & ctx_words) / len(sent_words) >= 0.4:
                supported += 1
        
        return round(supported / len(gt_sentences), 3)


class RAGASAnswerCorrectness:
    """
    RAGAS Answer Correctness: factual + semantic similarity between answer and ground truth.
    Combines: factual overlap (0.75 weight) + semantic overlap (0.25 weight)
    """
    name = "ragas_answer_correctness"
    
    def score(self, answer: str, ground_truth: str) -> float:
        a_words = set(w.lower() for w in answer.split() if len(w) > 3)
        g_words = set(w.lower() for w in ground_truth.split() if len(w) > 3)
        
        if not a_words or not g_words:
            return 0.0
        
        # F1 over content words
        common = len(a_words & g_words)
        precision = common / len(a_words)
        recall = common / len(g_words)
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        
        return round(f1, 3)


# ── RAGAS Dataset and Evaluation ──────────────────────────────────────────────
ragas_dataset = [
    {
        "question": "What is RAGAS and what metrics does it provide?",
        "answer": "RAGAS is a framework for evaluating RAG pipelines. It provides metrics including faithfulness, answer relevancy, context precision, and context recall to measure different aspects of retrieval-augmented generation quality.",
        "contexts": [
            "RAGAS (Retrieval Augmented Generation Assessment) evaluates RAG pipelines on faithfulness, answer relevancy, context precision, and context recall metrics.",
            "RAGAS was designed to provide standardized evaluation for LLM applications that use retrieval.",
        ],
        "ground_truth": "RAGAS evaluates RAG pipelines using faithfulness, answer relevancy, context precision, and context recall metrics.",
    },
    {
        "question": "How does DeepEval differ from RAGAS?",
        "answer": "DeepEval focuses on unit-test style evaluation for LLM outputs with metrics like hallucination detection and answer relevancy. RAGAS specializes in RAG pipeline evaluation with context-aware metrics. DeepEval has better CI/CD integration while RAGAS has more RAG-specific metrics.",
        "contexts": [
            "DeepEval is an open-source LLM evaluation framework that provides unit-test-style evaluation.",
            "RAGAS focuses specifically on RAG pipeline evaluation with faithfulness and context metrics.",
        ],
        "ground_truth": "DeepEval is for LLM unit testing while RAGAS specializes in RAG pipeline evaluation.",
    },
    {
        "question": "What is faithfulness in the context of RAGAS?",
        "answer": "Faithfulness in RAGAS measures whether the generated answer is grounded in the retrieved context. A high faithfulness score means the answer only contains information that can be inferred from the provided context, with no hallucinations.",
        "contexts": [
            "Faithfulness measures whether the agent's answer is grounded in the retrieved context. An unfaithful answer introduces information not present in the context.",
        ],
        "ground_truth": "Faithfulness measures if the answer is grounded in retrieved context, with no information beyond what the context supports.",
    },
]

# Run RAGAS evaluation
ragas_metrics = {
    "faithfulness": RAGASFaithfulness(),
    "answer_relevancy": RAGASAnswerRelevancy(),
    "context_precision": RAGASContextPrecision(),
    "context_recall": RAGASContextRecall(),
    "answer_correctness": RAGASAnswerCorrectness(),
}

rprint(Panel("[bold]8.2 RAGAS-Style RAG Pipeline Evaluation[/bold]", border_style="cyan"))

ragas_results_df = []
for item in ragas_dataset:
    row = {"question": item["question"][:40] + "..."}
    
    row["faithfulness"] = ragas_metrics["faithfulness"].score(item["answer"], item["contexts"])
    row["answer_relevancy"] = ragas_metrics["answer_relevancy"].score(item["question"], item["answer"])
    row["context_precision"] = ragas_metrics["context_precision"].score(item["question"], item["contexts"], item["ground_truth"])
    row["context_recall"] = ragas_metrics["context_recall"].score(item["ground_truth"], item["contexts"])
    row["answer_correctness"] = ragas_metrics["answer_correctness"].score(item["answer"], item["ground_truth"])
    row["ragas_score"] = round(np.mean([row[m] for m in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]]), 3)
    
    ragas_results_df.append(row)

df_ragas = pd.DataFrame(ragas_results_df)

table = Table(title="RAGAS Evaluation Results", header_style="bold green")
for col in df_ragas.columns:
    table.add_column(col[:20], max_width=22)

for _, r in df_ragas.iterrows():
    row_vals = []
    for col in df_ragas.columns:
        val = r[col]
        if isinstance(val, float):
            color = "green" if val >= 0.7 else "yellow" if val >= 0.5 else "red"
            row_vals.append(f"[{color}]{val:.3f}[/{color}]")
        else:
            row_vals.append(str(val)[:20])
    table.add_row(*row_vals)

console.print(table)

# Print RAGAS aggregate
rprint(f"\n[bold]RAGAS Score Averages:[/bold]")
metric_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall", "answer_correctness", "ragas_score"]
for col in metric_cols:
    avg = df_ragas[col].mean()
    color = "green" if avg >= 0.7 else "yellow" if avg >= 0.5 else "red"
    rprint(f"  {col:25s} [{color}]{avg:.3f}[/{color}]")

rprint("\n[bold]📌 Real RAGAS Usage (requires LLM API key):[/bold]")
rprint("""[dim]
  from ragas import evaluate
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
  from datasets import Dataset
  
  data = {
      "question": ["What is RAGAS?"],
      "answer": ["RAGAS evaluates RAG pipelines..."],
      "contexts": [["RAGAS is a framework for RAG evaluation..."]],
      "ground_truth": ["RAGAS is a RAG evaluation framework."],
  }
  dataset = Dataset.from_dict(data)
  
  result = evaluate(dataset, metrics=[faithfulness, answer_relevancy, context_precision, context_recall])
  print(result)  # DataFrame with per-row scores
[/dim]""")


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 8.2 RAGAS-Style RAG Pipeline Evaluation                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                             RAGAS Evaluation Results                                              
┏━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ question       ┃ faithfulness ┃ answer_releva… ┃ context_preci… ┃ context_recall ┃ answer_correc… ┃ ragas_score ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ What is RAGAS  │ 1.000        │ 0.156          │ 0.500          │ 1.000          │ 0.467          │ 0.664       │
│ and wh         │              │                │                │                │                │             │
│ How does       │ 0.333        │ 0.083          │ 0.000          │ 1.000          │ 0.312          │ 0.354       │
│ DeepEval di    │              │                │                │                │                │             │
│ What is        │ 0.500        │ 0.090          │ 1.000          │ 1.000          │ 0.471          │ 0.648       │
│ faithfulness   │              │                │                │                │                │             │
└────────────────┴──────────────┴────────────────┴────────────────┴────────────────┴────────────────┴─────────────┘

RAGAS Score Averages:

faithfulness              0.611

answer_relevancy          0.110

...

context_recall            1.000

answer_correctness        0.417

ragas_score               0.555

📌 Real RAGAS Usage (requires LLM API key):

  from ragas import evaluate
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
  from datasets import Dataset

  data = {
      "question": ["What is RAGAS?"],
      "answer": ["RAGAS evaluates RAG pipelines..."],
      "contexts": [["RAGAS is a framework for RAG evaluation..."]],
      "ground_truth": ["RAGAS is a RAG evaluation framework."],
  }
  dataset = Dataset.from_dict(data)

  result = evaluate(dataset, metrics=)
  print(result)  # DataFrame with per-row scores

In [22]:
# ─── 9.1 MULTI-TURN CONVERSATION EVALUATION ───────────────────────────────────
# Evaluating single turns is insufficient. We need to measure quality across turns:
# - Context retention: does agent remember earlier facts?
# - Consistency: no contradictions between turns?
# - Progressive improvement: later turns better than earlier?

@dataclass
class ConversationTurn:
    turn_id: int
    human: str
    agent: str
    tools_called: List[str] = field(default_factory=list)
    latency_ms: float = 0.0


class MultiTurnEvaluator:
    """Evaluates multi-turn conversation quality"""
    
    def context_retention_score(self, conversation: List[ConversationTurn]) -> float:
        """
        Does the agent reference earlier context in later turns?
        Heuristic: check if later turns mention words from earlier human turns.
        """
        if len(conversation) < 2:
            return 1.0
        
        scores = []
        for i in range(1, len(conversation)):
            # Gather all earlier context
            earlier_words = set()
            for prev in conversation[:i]:
                earlier_words.update(w.lower() for w in prev.human.split() if len(w) > 4)
                earlier_words.update(w.lower() for w in prev.agent.split() if len(w) > 4)
            
            # Check current agent response references earlier context
            current_words = set(w.lower() for w in conversation[i].agent.split() if len(w) > 4)
            retention = len(earlier_words & current_words) / len(earlier_words) if earlier_words else 1.0
            scores.append(min(1.0, retention * 2))  # scale up — even 50% overlap is good
        
        return round(np.mean(scores), 3) if scores else 1.0
    
    def consistency_score(self, conversation: List[ConversationTurn]) -> float:
        """
        Are agent responses internally consistent? 
        Simplified: check that agent doesn't contradict itself on key facts.
        Real version: use NLI model to detect entailment/contradiction.
        """
        agent_responses = [t.agent for t in conversation]
        
        # Build word frequency across all agent responses
        all_words = " ".join(agent_responses).lower().split()
        word_freq = {}
        for w in all_words:
            if len(w) > 4:
                word_freq[w] = word_freq.get(w, 0) + 1
        
        # High consistency if same key terms appear multiple times (coherent vocabulary)
        repeated_terms = sum(1 for count in word_freq.values() if count >= 2)
        total_terms = len(word_freq)
        
        return round(min(1.0, repeated_terms / max(1, total_terms / 2)), 3)
    
    def tool_consistency_score(self, conversation: List[ConversationTurn]) -> float:
        """Are the same tools used consistently for similar tasks?"""
        all_tool_calls = [tc for t in conversation for tc in t.tools_called]
        if len(all_tool_calls) < 2:
            return 1.0
        
        # Calculate variety (low variety = consistent tool use)
        unique_tools = set(all_tool_calls)
        variety_ratio = len(unique_tools) / len(all_tool_calls)
        # Moderate variety is good (not all same, not all different)
        return round(1.0 - abs(variety_ratio - 0.5), 3)
    
    def response_quality_trend(self, conversation: List[ConversationTurn]) -> Dict:
        """Does response quality improve over the conversation? (length proxy)"""
        lengths = [len(t.agent.split()) for t in conversation]
        
        if len(lengths) < 2:
            return {"trend": "insufficient_data", "slope": 0.0}
        
        # Compute slope of response length over turns
        x = np.arange(len(lengths))
        slope = np.polyfit(x, lengths, 1)[0]
        
        return {
            "trend": "improving" if slope > 0.5 else "declining" if slope < -0.5 else "stable",
            "slope": round(slope, 2),
            "avg_length": round(np.mean(lengths), 1),
        }
    
    def evaluate_full_conversation(self, conversation: List[ConversationTurn]) -> Dict:
        return {
            "context_retention": self.context_retention_score(conversation),
            "consistency": self.consistency_score(conversation),
            "tool_consistency": self.tool_consistency_score(conversation),
            "quality_trend": self.response_quality_trend(conversation),
            "total_turns": len(conversation),
            "total_tool_calls": sum(len(t.tools_called) for t in conversation),
        }


# ── Simulate a multi-turn conversation ────────────────────────────────────────
simulated_conversation = [
    ConversationTurn(1, "Hi, I need help with my insurance claim for policy POL-2024-8821.",
                     "Hello! I'll help you with your insurance claim for policy POL-2024-8821. I've retrieved your policy details.",
                     tools_called=["lookup_policy"], latency_ms=250),
    ConversationTurn(2, "The claim was rejected. What are my options?",
                     "For policy POL-2024-8821, since your claim was rejected, you have several options: 1) File a grievance with the insurer's grievance cell. 2) Escalate to IRDAI. 3) Approach the Insurance Ombudsman.",
                     tools_called=["web_search_mock"], latency_ms=310),
    ConversationTurn(3, "I want to submit a formal grievance. How do I do that?",
                     "I'll submit a formal grievance for policy POL-2024-8821 right away. Based on your earlier mention of the rejected claim, I'm categorizing this as 'claim_rejection'. Filing now...",
                     tools_called=["submit_grievance"], latency_ms=420),
    ConversationTurn(4, "What are the chances of winning?",
                     "Based on your grievance for policy POL-2024-8821 and the evidence for the rejected claim, I've run a winnability analysis. Your case has a 72% chance of resolution in your favor based on similar IRDAI precedents.",
                     tools_called=["check_winnability"], latency_ms=380),
]

mt_evaluator = MultiTurnEvaluator()
mt_results = mt_evaluator.evaluate_full_conversation(simulated_conversation)

rprint(Panel("[bold]9.1 Multi-Turn Conversation Evaluation[/bold]", border_style="cyan"))
for k, v in mt_results.items():
    if isinstance(v, dict):
        rprint(f"  [bold]{k}:[/bold] {v}")
    elif isinstance(v, float):
        color = "green" if v >= 0.7 else "yellow" if v >= 0.5 else "red"
        rprint(f"  [bold]{k:25s}[/bold] [{color}]{v:.3f}[/{color}]")
    else:
        rprint(f"  [bold]{k:25s}[/bold] {v}")


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 9.1 Multi-Turn Conversation Evaluation                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

context_retention         0.457

consistency               0.381

tool_consistency          0.500

quality_trend: {'trend': 'improving', 'slope': np.float64(5.3), 'avg_length': np.float64(26.8)}

total_turns               4

total_tool_calls          4

In [23]:
# ─── 9.2 MULTI-AGENT SYSTEM EVALUATION ───────────────────────────────────────
# Evaluating multi-agent systems requires additional metrics beyond single-agent:
# - Handoff quality: did agents correctly transfer context?
# - Supervisor efficiency: minimal routing decisions?
# - Specialization coverage: did each specialist contribute?
# - Collaboration coherence: do outputs fit together?

class MultiAgentEvaluator:
    """Evaluates multi-agent system performance"""
    
    def handoff_quality_score(self, handoff_history: List[str], worker_outputs: Dict[str, str]) -> float:
        """
        Measures if handoffs preserved context.
        Checks: each worker's output references context from the task.
        """
        if not worker_outputs:
            return 0.0
        
        task_words = set()
        for output in worker_outputs.values():
            task_words.update(w.lower() for w in output.split() if len(w) > 4)
        
        coherence_scores = []
        outputs_list = list(worker_outputs.values())
        for i in range(1, len(outputs_list)):
            prev_words = set(w.lower() for w in outputs_list[i-1].split() if len(w) > 4)
            curr_words = set(w.lower() for w in outputs_list[i].split() if len(w) > 4)
            overlap = len(prev_words & curr_words) / len(prev_words) if prev_words else 0
            coherence_scores.append(overlap)
        
        return round(np.mean(coherence_scores) if coherence_scores else 1.0, 3)
    
    def supervisor_efficiency_score(self, total_routing_decisions: int, num_workers: int) -> float:
        """Supervisor should route each worker once ideally"""
        optimal = num_workers + 1  # once per worker + final routing to END
        efficiency = optimal / max(total_routing_decisions, optimal)
        return round(min(1.0, efficiency), 3)
    
    def specialization_coverage(self, worker_outputs: Dict[str, str]) -> float:
        """Did all specialists contribute unique value?"""
        if len(worker_outputs) < 2:
            return 1.0
        
        outputs_list = list(worker_outputs.values())
        
        # Measure uniqueness: each output should have distinct vocabulary
        uniqueness_scores = []
        for i, output in enumerate(outputs_list):
            words = set(w.lower() for w in output.split() if len(w) > 4)
            others = set()
            for j, other in enumerate(outputs_list):
                if i != j:
                    others.update(w.lower() for w in other.split() if len(w) > 4)
            
            unique_contribution = len(words - others) / len(words) if words else 0
            uniqueness_scores.append(unique_contribution)
        
        return round(np.mean(uniqueness_scores), 3) if uniqueness_scores else 1.0
    
    def evaluate_multi_agent_run(self, run_data: Dict) -> Dict:
        handoff_hist = run_data.get("handoff_history", [])
        worker_outputs = run_data.get("worker_outputs", {})
        routing_decisions = run_data.get("supervisor_iterations", len(handoff_hist))
        
        return {
            "handoff_quality": self.handoff_quality_score(handoff_hist, worker_outputs),
            "supervisor_efficiency": self.supervisor_efficiency_score(
                routing_decisions, len(worker_outputs)
            ),
            "specialization_coverage": self.specialization_coverage(worker_outputs),
            "workers_used": len(worker_outputs),
            "routing_decisions": routing_decisions,
        }


# Evaluate the supervisor system run from Section 4
ma_evaluator = MultiAgentEvaluator()

ma_eval_input = {
    "handoff_history": ["triage→research_agent", "research→review"],  # from handoff demo
    "worker_outputs": multi_result.get("worker_outputs", {
        "researcher": "Evaluation frameworks include DeepEval, RAGAS, and custom Python evaluators.",
        "analyst": "Analysis: 10 words. Key themes: frameworks, evaluation, RAGAS.",
        "writer": "Executive Summary: Evaluation frameworks include DeepEval and RAGAS.",
    }),
    "supervisor_iterations": multi_result.get("supervisor_iterations", 4),
}

ma_results = ma_evaluator.evaluate_multi_agent_run(ma_eval_input)

rprint(Panel("[bold]9.2 Multi-Agent System Evaluation[/bold]", border_style="cyan"))
for k, v in ma_results.items():
    if isinstance(v, float):
        color = "green" if v >= 0.6 else "yellow" if v >= 0.4 else "red"
        rprint(f"  [bold]{k:30s}[/bold] [{color}]{v:.3f}[/{color}]")
    else:
        rprint(f"  [bold]{k:30s}[/bold] {v}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Multi-Agent System Evaluation Dashboard", fontweight="bold", fontsize=14)

metric_names = ["handoff_quality", "supervisor_efficiency", "specialization_coverage"]
metric_values = [ma_results[m] for m in metric_names]
colors_bar = ["#2ecc71" if v >= 0.6 else "#f39c12" if v >= 0.4 else "#e74c3c" for v in metric_values]

axes[0].barh(metric_names, metric_values, color=colors_bar)
axes[0].set_xlim(0, 1)
axes[0].axvline(x=0.7, color="gray", linestyle="--", alpha=0.5, label="Threshold (0.7)")
axes[0].set_title("Multi-Agent Metrics", fontweight="bold")
axes[0].set_xlabel("Score")
axes[0].legend(fontsize=8)
for i, (v, name) in enumerate(zip(metric_values, metric_names)):
    axes[0].text(v + 0.02, i, f"{v:.3f}", va="center", fontsize=9)

# Worker contribution
workers = list(ma_eval_input["worker_outputs"].keys())
word_counts = [len(v.split()) for v in ma_eval_input["worker_outputs"].values()]
axes[1].bar(workers, word_counts, color=["#3498db", "#9b59b6", "#e67e22"][:len(workers)])
axes[1].set_title("Worker Output Volume", fontweight="bold")
axes[1].set_ylabel("Word Count")
axes[1].set_xlabel("Agent")

plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/multi_agent_eval.png", dpi=120, bbox_inches="tight")
plt.close()
rprint("[green]✅ Multi-agent eval chart saved[/green]")


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 9.2 Multi-Agent System Evaluation                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

handoff_quality                0.000

supervisor_efficiency          1.000

specialization_coverage        0.462

workers_used                   3

routing_decisions              4

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/user-data/outputs/multi_agent_eval.png'

In [ ]:
# ─── 9.3 LATENCY & COST EVALUATION ───────────────────────────────────────────
# Production agents must meet latency SLAs and cost budgets.
# This section builds an observability layer for these non-quality metrics.

@dataclass
class AgentRunMetrics:
    run_id: str
    task: str
    total_latency_ms: float
    llm_calls: int
    tool_calls: int
    input_tokens: int
    output_tokens: int
    cost_usd: float  # estimated
    success: bool

    @property
    def tokens_per_ms(self) -> float:
        return self.output_tokens / self.total_latency_ms if self.total_latency_ms else 0
    
    @property
    def cost_per_output_token(self) -> float:
        return self.cost_usd / max(1, self.output_tokens)


class LatencyCostEvaluator:
    """Evaluate agent runs on latency and cost dimensions"""
    
    def __init__(self, 
                 p50_sla_ms: float = 2000,
                 p95_sla_ms: float = 5000,
                 cost_budget_usd: float = 0.05):
        self.p50_sla = p50_sla_ms
        self.p95_sla = p95_sla_ms
        self.budget = cost_budget_usd
    
    def sla_compliance(self, runs: List[AgentRunMetrics]) -> Dict:
        latencies = sorted([r.total_latency_ms for r in runs])
        n = len(latencies)
        p50 = latencies[int(n * 0.5)] if latencies else 0
        p95 = latencies[int(n * 0.95)] if latencies else 0
        p99 = latencies[int(n * 0.99)] if latencies else 0
        
        under_p50_sla = sum(1 for r in runs if r.total_latency_ms <= self.p50_sla) / n
        under_p95_sla = sum(1 for r in runs if r.total_latency_ms <= self.p95_sla) / n
        
        return {
            "p50_ms": round(p50, 0),
            "p95_ms": round(p95, 0),
            "p99_ms": round(p99, 0),
            "p50_sla_compliance": round(under_p50_sla, 3),
            "p95_sla_compliance": round(under_p95_sla, 3),
            "sla_target_p50": self.p50_sla,
            "sla_target_p95": self.p95_sla,
        }
    
    def cost_analysis(self, runs: List[AgentRunMetrics]) -> Dict:
        costs = [r.cost_usd for r in runs]
        return {
            "total_cost_usd": round(sum(costs), 4),
            "avg_cost_per_run": round(np.mean(costs), 5),
            "max_cost_run": round(max(costs), 5),
            "budget_usd": self.budget,
            "budget_compliance": round(sum(1 for c in costs if c <= self.budget) / len(costs), 3),
            "total_input_tokens": sum(r.input_tokens for r in runs),
            "total_output_tokens": sum(r.output_tokens for r in runs),
        }
    
    def efficiency_score(self, runs: List[AgentRunMetrics]) -> float:
        """Combined efficiency: normalize latency and cost to [0,1]"""
        lat_scores = [min(1.0, self.p95_sla / max(1, r.total_latency_ms)) for r in runs]
        cost_scores = [min(1.0, self.budget / max(0.0001, r.cost_usd)) for r in runs]
        return round(np.mean([l * c for l, c in zip(lat_scores, cost_scores)]), 3)


# ── Simulate 50 agent runs with realistic latency/cost distribution ─────────
def simulate_agent_run(task: str, complexity: str = "medium") -> AgentRunMetrics:
    latency_params = {"simple": (500, 200), "medium": (2000, 800), "complex": (4500, 1500)}
    mu, sigma = latency_params.get(complexity, (2000, 800))
    latency = max(100, np.random.normal(mu, sigma))
    
    llm_calls = random.randint(1, 4)
    tool_calls = random.randint(0, 3)
    input_tokens = random.randint(200, 2000)
    output_tokens = random.randint(50, 800)
    
    # Cost: estimate at $0.003/1K input + $0.015/1K output (GPT-4 approx)
    cost = (input_tokens * 0.003 + output_tokens * 0.015) / 1000
    
    return AgentRunMetrics(
        run_id=f"run-{uuid.uuid4().hex[:6]}",
        task=task,
        total_latency_ms=round(latency, 1),
        llm_calls=llm_calls,
        tool_calls=tool_calls,
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        cost_usd=round(cost, 6),
        success=random.random() > 0.05,  # 95% success rate
    )


TASK_TYPES = [
    ("Simple greeting response", "simple"),
    ("Policy lookup and summary", "medium"),
    ("Full grievance resolution pipeline", "complex"),
    ("Multi-document analysis", "complex"),
    ("Quick FAQ answer", "simple"),
]

np.random.seed(42)
runs = []
for _ in range(50):
    task, complexity = random.choice(TASK_TYPES)
    runs.append(simulate_agent_run(task, complexity))

evaluator_lc = LatencyCostEvaluator(p50_sla_ms=2000, p95_sla_ms=5000, cost_budget_usd=0.05)
sla = evaluator_lc.sla_compliance(runs)
costs = evaluator_lc.cost_analysis(runs)
eff = evaluator_lc.efficiency_score(runs)

# Dashboard
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Agent Latency & Cost Dashboard (50 runs)", fontweight="bold", fontsize=14)

# 1. Latency distribution
latencies = [r.total_latency_ms for r in runs]
axes[0, 0].hist(latencies, bins=20, color="#3498db", alpha=0.7, edgecolor="white")
axes[0, 0].axvline(sla["p50_ms"], color="#e74c3c", linestyle="--", label=f"P50: {sla['p50_ms']:.0f}ms")
axes[0, 0].axvline(sla["p95_ms"], color="#f39c12", linestyle="--", label=f"P95: {sla['p95_ms']:.0f}ms")
axes[0, 0].axvline(2000, color="gray", linestyle=":", label="P50 SLA (2s)")
axes[0, 0].set_title("Latency Distribution"); axes[0, 0].set_xlabel("ms"); axes[0, 0].legend(fontsize=8)

# 2. Cost distribution
costs_list = [r.cost_usd for r in runs]
axes[0, 1].hist(costs_list, bins=20, color="#2ecc71", alpha=0.7, edgecolor="white")
axes[0, 1].axvline(evaluator_lc.budget, color="#e74c3c", linestyle="--", label=f"Budget: ${evaluator_lc.budget}")
axes[0, 1].axvline(np.mean(costs_list), color="#9b59b6", linestyle="-.", label=f"Mean: ${np.mean(costs_list):.4f}")
axes[0, 1].set_title("Cost Distribution"); axes[0, 1].set_xlabel("USD"); axes[0, 1].legend(fontsize=8)

# 3. SLA compliance
sla_data = {"P50 SLA\n(2s)": sla["p50_sla_compliance"], "P95 SLA\n(5s)": sla["p95_sla_compliance"]}
bars = axes[1, 0].bar(sla_data.keys(), sla_data.values(), color=["#2ecc71" if v >= 0.9 else "#e74c3c" for v in sla_data.values()])
axes[1, 0].set_ylim(0, 1.1); axes[1, 0].axhline(0.95, color="gray", linestyle="--", label="Target (95%)")
axes[1, 0].set_title("SLA Compliance Rate"); axes[1, 0].set_ylabel("Fraction")
axes[1, 0].legend()
for bar, val in zip(bars, sla_data.values()):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, val + 0.02, f"{val:.1%}", ha="center", fontweight="bold")

# 4. Scatter: latency vs cost (complexity)
scatter_colors = [
    "#2ecc71" if r.total_latency_ms < 1000 else "#f39c12" if r.total_latency_ms < 3000 else "#e74c3c"
    for r in runs
]
axes[1, 1].scatter([r.total_latency_ms for r in runs], [r.cost_usd for r in runs], 
                   c=scatter_colors, alpha=0.6, s=40)
axes[1, 1].set_title("Latency vs Cost")
axes[1, 1].set_xlabel("Latency (ms)"); axes[1, 1].set_ylabel("Cost (USD)")
patches = [mpatches.Patch(color=c, label=l) for c, l in [("#2ecc71", "Fast <1s"), ("#f39c12", "Medium"), ("#e74c3c", "Slow >3s")]]
axes[1, 1].legend(handles=patches, fontsize=8)

plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/latency_cost_dashboard.png", dpi=120, bbox_inches="tight")
plt.close()

rprint(Panel(
    f"[bold]SLA Compliance:[/bold]\n"
    f"  P50 ({sla['sla_target_p50']:.0f}ms SLA):  {sla['p50_sla_compliance']:.1%}\n"
    f"  P95 ({sla['sla_target_p95']:.0f}ms SLA):  {sla['p95_sla_compliance']:.1%}\n\n"
    f"[bold]Cost Analysis:[/bold]\n"
    f"  Total cost (50 runs): ${costs['total_cost_usd']:.4f}\n"
    f"  Avg per run: ${costs['avg_cost_per_run']:.5f}\n"
    f"  Budget compliance: {costs['budget_compliance']:.1%}\n"
    f"  Total tokens: {costs['total_input_tokens']:,} in / {costs['total_output_tokens']:,} out\n\n"
    f"[bold green]Overall Efficiency Score: {eff:.3f}[/bold green]",
    title="[cyan]9.3 Latency & Cost Evaluation[/cyan]",
    border_style="cyan"
))


In [24]:
# ─── 10.1 PRODUCTION EVALUATION PIPELINE ─────────────────────────────────────
# The full production evaluation layer:
# 1. Regression test suite (catches regressions)
# 2. LLM-as-judge (qualitative scoring)
# 3. Composite scoring with weights
# 4. CI/CD integration (pytest)
# 5. Eval dashboard
#
# This is the CAPSTONE of the notebook.

# ════════════════════════════════════════════════════════════════
# PRODUCTION EVALUATION ARCHITECTURE
#
#  ┌────────────────────────────────────────────────────────────┐
#  │                PRODUCTION EVAL PIPELINE                    │
#  │                                                            │
#  │  [Agent Run] → [Trace Collector] → [Eval Router]          │
#  │                                          │                 │
#  │                        ┌─────────────────┼───────────────┐│
#  │                        ▼                 ▼               ▼││
#  │              [Component Evals]  [System Evals]  [Cost Evals]│
#  │              (faithfulness,     (task success,  (latency,    │
#  │               tool accuracy,    E2E correctness) tokens,     │
#  │               safety)                           cost/run)    │
#  │                        └─────────────────┼───────────────┘│
#  │                                          ▼                 │
#  │                             [Composite Score]              │
#  │                                  │                         │
#  │                    ┌─────────────┴──────────┐              │
#  │                    ▼                        ▼              │
#  │           [Pass/Fail Decision]    [Dashboard + Alerts]     │
#  │                    │                                       │
#  │                    ▼                                       │
#  │            [CI/CD Gate]  ← pytest integration              │
#  └────────────────────────────────────────────────────────────┘
# ════════════════════════════════════════════════════════════════

class ProductionEvalConfig:
    """Configuration for production evaluation"""
    
    # Metric weights for composite score
    METRIC_WEIGHTS = {
        "faithfulness": 0.25,
        "answer_relevancy": 0.20,
        "tool_accuracy": 0.20,
        "safety": 0.15,
        "latency": 0.10,
        "trajectory_accuracy": 0.10,
    }
    
    # SLAs
    LATENCY_P95_MS = 5000
    LATENCY_P50_MS = 2000
    COST_BUDGET_USD = 0.10
    
    # Thresholds
    COMPONENT_THRESHOLD = 0.70  # per-metric pass threshold
    COMPOSITE_THRESHOLD = 0.75  # overall score to pass
    REGRESSION_THRESHOLD = 0.05  # max allowed degradation vs baseline
    
    # Safety is non-negotiable
    SAFETY_HARD_LIMIT = 0.90


class EvalOrchestrator:
    """
    Top-level evaluation orchestrator.
    Collects all metrics, computes composite score, makes pass/fail decision.
    """
    
    def __init__(self, config: ProductionEvalConfig = None):
        self.config = config or ProductionEvalConfig()
        self.baseline_scores: Optional[Dict] = None
        self.eval_history: List[Dict] = []
    
    def evaluate_agent_output(self, 
                               question: str, 
                               answer: str, 
                               contexts: List[str],
                               expected_tools: List[str],
                               actual_tools: List[str],
                               expected_trajectory: List[str],
                               actual_trajectory: List[str],
                               latency_ms: float,
                               ) -> Dict:
        """Run all metrics and return composite result"""
        
        # ── Component metrics ──────────────────────────────────────────
        faith_metric = RAGASFaithfulness()
        rel_metric = RAGASAnswerRelevancy()
        safety_metric = SafetyMetric()
        traj_metric = TrajectoryAccuracyMetric()
        tool_metric = BasicAgentEvaluator()
        
        faith_score = faith_metric.score(answer, contexts)
        rel_score = rel_metric.score(question, answer)
        safety_score = safety_metric.score(answer)
        
        traj_result = traj_metric.evaluate(
            expected_trajectory=expected_trajectory,
            actual_trajectory=actual_trajectory,
        )
        
        tool_tc = AgentTestCase(
            test_id="runtime", input=question, 
            expected_output=answer, expected_tools=expected_tools,
            actual_output=answer, actual_tools=actual_tools, latency_ms=latency_ms
        )
        tool_result = tool_metric.tool_accuracy(tool_tc)
        lat_result = tool_metric.latency_check(tool_tc, sla_ms=self.config.LATENCY_P95_MS)
        
        scores = {
            "faithfulness": faith_score,
            "answer_relevancy": rel_score,
            "tool_accuracy": tool_result.score,
            "safety": safety_score,
            "latency": lat_result.score,
            "trajectory_accuracy": traj_result.score,
        }
        
        # ── Composite score (weighted average) ─────────────────────────
        weights = self.config.METRIC_WEIGHTS
        composite = sum(scores[m] * weights[m] for m in scores if m in weights)
        
        # ── Safety hard limit ──────────────────────────────────────────
        safety_fail = scores["safety"] < self.config.SAFETY_HARD_LIMIT
        
        # ── Pass/Fail ─────────────────────────────────────────────────
        overall_pass = (
            composite >= self.config.COMPOSITE_THRESHOLD and
            not safety_fail and
            all(scores[m] >= self.config.COMPONENT_THRESHOLD 
                for m in scores if m != "latency")  # latency is advisory
        )
        
        result = {
            "scores": scores,
            "composite_score": round(composite, 3),
            "passed": overall_pass,
            "safety_hard_fail": safety_fail,
            "recommendations": self._generate_recommendations(scores, overall_pass),
            "timestamp": datetime.now().isoformat(),
        }
        
        self.eval_history.append(result)
        return result
    
    def _generate_recommendations(self, scores: Dict, passed: bool) -> List[str]:
        recs = []
        if scores.get("faithfulness", 1) < 0.7:
            recs.append("⚠️  Low faithfulness: improve context retrieval quality or reduce LLM creativity")
        if scores.get("answer_relevancy", 1) < 0.7:
            recs.append("⚠️  Low relevancy: refine system prompt to stay on-topic")
        if scores.get("tool_accuracy", 1) < 0.7:
            recs.append("⚠️  Poor tool selection: improve tool descriptions or add tool routing logic")
        if scores.get("trajectory_accuracy", 1) < 0.7:
            recs.append("⚠️  Off-track trajectory: review routing conditions in graph")
        if scores.get("safety", 1) < 0.9:
            recs.append("🚨 SAFETY CONCERN: add safety guardrails (Guardrails AI, NeMo Guardrails)")
        if scores.get("latency", 1) < 0.8:
            recs.append("⚡ Latency SLA breach: consider caching, smaller model, or parallel execution")
        if not recs:
            recs.append("✅ All metrics within acceptable range")
        return recs
    
    def regression_check(self, current_scores: Dict, baseline: Dict) -> Dict:
        """Compare current eval against baseline — detect regressions"""
        regressions = {}
        improvements = {}
        
        for metric, current in current_scores.items():
            if metric in baseline:
                delta = current - baseline[metric]
                if delta < -self.config.REGRESSION_THRESHOLD:
                    regressions[metric] = {
                        "baseline": baseline[metric],
                        "current": current,
                        "delta": round(delta, 3),
                    }
                elif delta > self.config.REGRESSION_THRESHOLD:
                    improvements[metric] = round(delta, 3)
        
        return {
            "regressions_detected": len(regressions) > 0,
            "regression_count": len(regressions),
            "regressions": regressions,
            "improvements": improvements,
            "ci_gate": "FAIL" if regressions else "PASS",
        }


# ── Run the full production evaluation ───────────────────────────────────────
orchestrator = EvalOrchestrator()

production_test_cases = [
    {
        "question": "What is RAGAS and how does it evaluate RAG systems?",
        "answer": "RAGAS evaluates RAG pipelines using faithfulness, answer relevancy, context precision, and context recall metrics. It measures whether answers are grounded in retrieved context.",
        "contexts": ["RAGAS (Retrieval Augmented Generation Assessment) evaluates RAG pipelines on faithfulness, answer relevancy, context precision, and context recall metrics."],
        "expected_tools": ["web_search_mock"],
        "actual_tools": ["web_search_mock"],
        "expected_trajectory": ["retrieve", "generate", "grade"],
        "actual_trajectory": ["retrieve", "generate", "grade"],
        "latency_ms": 1200,
        "case_name": "RAG evaluation query",
    },
    {
        "question": "Process the insurance grievance for claim rejection",
        "answer": "Grievance submitted successfully. Your case GR-ABC123 has been filed under claim_rejection category. Estimated resolution: 15 days. Based on IRDAI analysis, winnability score is 0.78.",
        "contexts": ["IRDAI regulations require grievance resolution within 15 days. Claim rejection is a valid grievance category."],
        "expected_tools": ["lookup_policy", "submit_grievance", "check_winnability"],
        "actual_tools": ["lookup_policy", "submit_grievance", "check_winnability"],
        "expected_trajectory": ["retrieve_policy", "submit_grievance", "predict_winnability", "respond"],
        "actual_trajectory": ["retrieve_policy", "submit_grievance", "predict_winnability", "respond"],
        "latency_ms": 3400,
        "case_name": "Insurance grievance (complex)",
    },
    {
        "question": "What is the weather today?",
        "answer": "I cannot determine the exact weather without knowing your location. Please provide your city name and I will use the weather tool to fetch current conditions.",
        "contexts": [],
        "expected_tools": ["get_weather"],
        "actual_tools": [],  # ← agent didn't call the tool — bug!
        "expected_trajectory": ["ask_location", "get_weather", "respond"],
        "actual_trajectory": ["respond"],
        "latency_ms": 6200,  # ← SLA breach
        "case_name": "Weather query (with bugs)",
    },
]

rprint(Panel("[bold]10.1 Production Evaluation Pipeline — Full Run[/bold]", border_style="green"))

full_results = []
for tc in production_test_cases:
    result = orchestrator.evaluate_agent_output(
        question=tc["question"],
        answer=tc["answer"],
        contexts=tc["contexts"],
        expected_tools=tc["expected_tools"],
        actual_tools=tc["actual_tools"],
        expected_trajectory=tc["expected_trajectory"],
        actual_trajectory=tc["actual_trajectory"],
        latency_ms=tc["latency_ms"],
    )
    full_results.append({"name": tc["case_name"], **result})
    
    color = "green" if result["passed"] else "red"
    rprint(f"\n[bold]{tc['case_name']}[/bold]")
    rprint(f"  Composite: [{color}]{result['composite_score']:.3f}[/{color}]  |  " +
          f"Pass: [{color}]{'✅ YES' if result['passed'] else '❌ NO'}[/{color}]")
    
    for metric, score in result["scores"].items():
        color_m = "green" if score >= 0.7 else "yellow" if score >= 0.5 else "red"
        rprint(f"    {metric:22s}: [{color_m}]{score:.3f}[/{color_m}]")
    
    for rec in result["recommendations"]:
        rprint(f"  [yellow]{rec}[/yellow]")


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 10.1 Production Evaluation Pipeline — Full Run                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RAG evaluation query

Composite: 0.690  |  Pass: ❌ NO

faithfulness          : 0.500

answer_relevancy      : 0.073

...

⚠️  Low faithfulness: improve context retrieval quality or reduce LLM creativity

⚠️  Low relevancy: refine system prompt to stay on-topic

⚠️  Poor tool selection: improve tool descriptions or add tool routing logic

⚠️  Off-track trajectory: review routing conditions in graph

⚡ Latency SLA breach: consider caching, smaller model, or parallel execution

In [25]:
# ─── 10.2 REGRESSION TESTING & CI/CD INTEGRATION ─────────────────────────────
# Show how agent evaluation integrates into CI/CD pipelines.
# Pattern: run eval suite on every PR → fail build if regression detected.

# ── Define a regression test suite ────────────────────────────────────────────
class AgentEvalTestSuite:
    """
    Pytest-compatible test suite for agent evaluation.
    
    Real usage in CI/CD:
        pytest tests/agent_eval.py --eval-threshold=0.75
    """
    
    def __init__(self, agent, orchestrator: EvalOrchestrator, baseline: Dict = None):
        self.agent = agent
        self.orchestrator = orchestrator
        self.baseline = baseline
        self.test_results = []
    
    def run_test(self, test_case: Dict, test_name: str) -> bool:
        """Run a single test and return pass/fail"""
        result = self.orchestrator.evaluate_agent_output(**test_case)
        passed = result["passed"]
        self.test_results.append({"name": test_name, "passed": passed, "score": result["composite_score"]})
        return passed
    
    def run_suite(self, test_cases: List[Dict]) -> Dict:
        """Run all tests. Returns suite-level pass/fail."""
        passed_count = 0
        for i, tc in enumerate(test_cases):
            name = tc.pop("test_name", f"test_{i+1}")
            passed = self.run_test(tc, name)
            if passed:
                passed_count += 1
        
        pass_rate = passed_count / len(test_cases) if test_cases else 0
        suite_passed = pass_rate >= 0.80  # 80% of tests must pass
        
        return {
            "total": len(test_cases),
            "passed": passed_count,
            "failed": len(test_cases) - passed_count,
            "pass_rate": round(pass_rate, 3),
            "suite_passed": suite_passed,
            "ci_gate": "✅ PASS" if suite_passed else "❌ FAIL — build blocked",
        }


# ── Baseline Scores (imagine these are from last month's model version) ──────
BASELINE_SCORES = {
    "faithfulness": 0.82,
    "answer_relevancy": 0.78,
    "tool_accuracy": 0.85,
    "safety": 0.98,
    "latency": 0.90,
    "trajectory_accuracy": 0.80,
}

# ── Current model scores from our eval run ────────────────────────────────────
current_scores_avg = {}
metric_names_all = ["faithfulness", "answer_relevancy", "tool_accuracy", "safety", "latency", "trajectory_accuracy"]
for m in metric_names_all:
    current_scores_avg[m] = round(np.mean([r["scores"].get(m, 0.5) for r in orchestrator.eval_history]), 3)

# ── Check for regressions ──────────────────────────────────────────────────────
reg_result = orchestrator.regression_check(current_scores_avg, BASELINE_SCORES)

rprint(Panel("[bold]10.2 Regression Check & CI/CD Gate[/bold]", border_style="cyan"))
rprint(f"\n[bold]Baseline vs Current Scores:[/bold]")

reg_table = Table(header_style="bold")
reg_table.add_column("Metric"); reg_table.add_column("Baseline"); reg_table.add_column("Current"); reg_table.add_column("Delta"); reg_table.add_column("Status")

for metric in metric_names_all:
    baseline_v = BASELINE_SCORES.get(metric, 0)
    current_v = current_scores_avg.get(metric, 0)
    delta = current_v - baseline_v
    
    is_regression = metric in reg_result.get("regressions", {})
    is_improvement = metric in reg_result.get("improvements", {})
    
    status = "🔴 REGRESSION" if is_regression else "🟢 IMPROVED" if is_improvement else "⚪ STABLE"
    delta_color = "red" if delta < -0.05 else "green" if delta > 0.05 else "white"
    
    reg_table.add_row(
        metric,
        f"{baseline_v:.3f}",
        f"{current_v:.3f}",
        f"[{delta_color}]{delta:+.3f}[/{delta_color}]",
        status,
    )

console.print(reg_table)

gate_color = "green" if reg_result["ci_gate"] == "PASS" else "red"
rprint(f"\n[bold]CI/CD Gate Result:[/bold] [{gate_color}]{reg_result['ci_gate']}[/{gate_color}]")
rprint(f"  Regressions: {reg_result['regression_count']} | Improvements: {len(reg_result.get('improvements', {}))}")

# ── Show pytest pattern ────────────────────────────────────────────────────────
rprint("\n[bold]📌 CI/CD Integration Pattern (pytest):[/bold]")
rprint("""[dim]
# tests/test_agent_eval.py
import pytest
from your_agent import build_agent
from eval_framework import EvalOrchestrator, ProductionEvalConfig

@pytest.fixture
def agent():
    return build_agent()

@pytest.fixture  
def evaluator():
    return EvalOrchestrator(ProductionEvalConfig())

@pytest.mark.agent_eval
def test_faithfulness_above_threshold(agent, evaluator):
    result = agent.invoke({"messages": [...], ...})
    eval_result = evaluator.evaluate_agent_output(...)
    assert eval_result["scores"]["faithfulness"] >= 0.70, \
        f"Faithfulness {eval_result['scores']['faithfulness']:.3f} below threshold"

@pytest.mark.agent_eval
def test_no_safety_violations(agent, evaluator):
    result = agent.invoke(...)
    eval_result = evaluator.evaluate_agent_output(...)
    assert eval_result["scores"]["safety"] >= 0.90, "Safety violation detected!"

@pytest.mark.agent_eval
@pytest.mark.parametrize("test_case", REGRESSION_TEST_SUITE)
def test_no_regression(agent, evaluator, test_case, baseline_scores):
    eval_result = evaluator.evaluate_agent_output(**test_case)
    reg = evaluator.regression_check(eval_result["scores"], baseline_scores)
    assert not reg["regressions_detected"], f"Regression: {reg['regressions']}"

# Run with: pytest tests/test_agent_eval.py -m agent_eval -v
[/dim]""")


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 10.2 Regression Check & CI/CD Gate                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Baseline vs Current Scores:

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Metric              ┃ Baseline ┃ Current ┃ Delta  ┃ Status        ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ faithfulness        │ 0.820    │ 0.167   │ -0.653 │ 🔴 REGRESSION │
│ answer_relevancy    │ 0.780    │ 0.075   │ -0.705 │ 🔴 REGRESSION │
│ tool_accuracy       │ 0.850    │ 0.667   │ -0.183 │ 🔴 REGRESSION │
│ safety              │ 0.980    │ 1.000   │ +0.020 │ ⚪ STABLE     │
│ latency             │ 0.900    │ 0.920   │ +0.020 │ ⚪ STABLE     │
│ trajectory_accuracy │ 0.800    │ 0.778   │ -0.022 │ ⚪ STABLE     │
└─────────────────────┴──────────┴─────────┴────────┴───────────────┘

CI/CD Gate Result: FAIL

Regressions: 3 | Improvements: 0

📌 CI/CD Integration Pattern (pytest):

# tests/test_agent_eval.py
import pytest
from your_agent import build_agent
from eval_framework import EvalOrchestrator, ProductionEvalConfig

@pytest.fixture
def agent():
    return build_agent()

@pytest.fixture  
def evaluator():
    return EvalOrchestrator(ProductionEvalConfig())

@pytest.mark.agent_eval
def test_faithfulness_above_threshold(agent, evaluator):
    result = agent.invoke({"messages": [...], ...})
    eval_result = evaluator.evaluate_agent_output(...)
    assert eval_result["scores"]["faithfulness"] >= 0.70,         f"Faithfulness 
{eval_result['scores']['faithfulness']:.3f} below threshold"

@pytest.mark.agent_eval
def test_no_safety_violations(agent, evaluator):
    result = agent.invoke(...)
    eval_result = evaluator.evaluate_agent_output(...)
    assert eval_result["scores"]["safety"] >= 0.90, "Safety violation detected!"

@pytest.mark.agent_eval
@pytest.mark.parametrize("test_case", REGRESSION_TEST_SUITE)
def test_no_regression(agent, evaluator, test_case, baseline_scores):
    eval_result = evaluator.evaluate_agent_output(**test_case)
    reg = evaluator.regression_check(eval_result["scores"], baseline_scores)
    assert not reg["regressions_detected"], f"Regression: {reg['regressions']}"

# Run with: pytest tests/test_agent_eval.py -m agent_eval -v

In [26]:
# ─── 10.3 MASTER EVALUATION DASHBOARD ────────────────────────────────────────
# Final visualization: comprehensive dashboard showing all evaluation dimensions

fig = plt.figure(figsize=(18, 14))
fig.suptitle("Agent Evaluation Master Dashboard", fontweight="bold", fontsize=16, y=0.98)

gs = fig.add_gridspec(3, 3, hspace=0.45, wspace=0.35)

# ── Panel 1: Composite scores per test case ────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
case_names = [r["name"][:35] for r in full_results]
composite_scores = [r["composite_score"] for r in full_results]
pass_flags = [r["passed"] for r in full_results]
bar_colors = ["#2ecc71" if p else "#e74c3c" for p in pass_flags]

bars = ax1.bar(case_names, composite_scores, color=bar_colors, edgecolor="white", linewidth=1.5)
ax1.axhline(y=orchestrator.config.COMPOSITE_THRESHOLD, color="#f39c12", linestyle="--", linewidth=2, label=f"Threshold ({orchestrator.config.COMPOSITE_THRESHOLD})")
ax1.set_ylim(0, 1.1); ax1.set_ylabel("Composite Score", fontsize=11)
ax1.set_title("Composite Evaluation Score by Test Case", fontsize=12, fontweight="bold")
ax1.legend(fontsize=10)
for bar, score, passed in zip(bars, composite_scores, pass_flags):
    ax1.text(bar.get_x() + bar.get_width()/2, score + 0.02, f"{score:.3f}", ha="center", fontsize=10, fontweight="bold")
    ax1.text(bar.get_x() + bar.get_width()/2, -0.08, "✅" if passed else "❌", ha="center", fontsize=14)

# ── Panel 2: Radar chart of metric scores (first test case) ──────────────────
ax2 = fig.add_subplot(gs[1, 0], polar=True)
radar_metrics = list(metric_names_all)
n = len(radar_metrics)
angles = [i / n * 2 * np.pi for i in range(n)] + [0]
values_1 = [full_results[0]["scores"].get(m, 0) for m in radar_metrics] + [full_results[0]["scores"].get(radar_metrics[0], 0)]

ax2.plot(angles, values_1, "o-", linewidth=2, color="#3498db", label=full_results[0]["name"][:20])
ax2.fill(angles, values_1, alpha=0.25, color="#3498db")
ax2.set_xticks(angles[:-1])
ax2.set_xticklabels([m.replace("_", "\n") for m in radar_metrics], size=7)
ax2.set_ylim(0, 1); ax2.axhline(y=0.7, color="gray", linewidth=0.5, linestyle="--")
ax2.set_title("Metric Profile\n(Test Case 1)", fontsize=10, fontweight="bold", pad=15)

# ── Panel 3: Metric heatmap across test cases ─────────────────────────────────
ax3 = fig.add_subplot(gs[1, 1:])
heatmap_data = np.array([
    [r["scores"].get(m, 0) for m in metric_names_all]
    for r in full_results
])
im = ax3.imshow(heatmap_data, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
ax3.set_xticks(range(len(metric_names_all)))
ax3.set_xticklabels([m.replace("_", "\n") for m in metric_names_all], fontsize=8, rotation=0, ha="center")
ax3.set_yticks(range(len(full_results)))
ax3.set_yticklabels([r["name"][:25] for r in full_results], fontsize=8)
ax3.set_title("Metric Heatmap Across Test Cases", fontsize=11, fontweight="bold")
plt.colorbar(im, ax=ax3, shrink=0.8, label="Score")
for i in range(len(full_results)):
    for j in range(len(metric_names_all)):
        val = heatmap_data[i, j]
        ax3.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8,
                color="white" if val < 0.5 else "black")

# ── Panel 4: Baseline regression ─────────────────────────────────────────────
ax4 = fig.add_subplot(gs[2, 0])
baseline_vals = [BASELINE_SCORES.get(m, 0) for m in metric_names_all]
current_vals = [current_scores_avg.get(m, 0) for m in metric_names_all]
x_pos = np.arange(len(metric_names_all))
width = 0.35
ax4.bar(x_pos - width/2, baseline_vals, width, label="Baseline", color="#3498db", alpha=0.7)
ax4.bar(x_pos + width/2, current_vals, width, label="Current", color="#2ecc71", alpha=0.7)
ax4.set_xticks(x_pos)
ax4.set_xticklabels([m[:8] for m in metric_names_all], rotation=45, ha="right", fontsize=7)
ax4.set_ylim(0, 1.1); ax4.set_title("Baseline vs Current", fontsize=10, fontweight="bold")
ax4.legend(fontsize=8); ax4.set_ylabel("Score")

# ── Panel 5: Latency percentiles ─────────────────────────────────────────────
ax5 = fig.add_subplot(gs[2, 1])
p_labels = ["P50", "P75", "P90", "P95", "P99"]
p_vals = [np.percentile(latencies, int(p[1:])) for p in p_labels]
sla_refs = [2000, 2000, 5000, 5000, 5000]
ax5.bar(p_labels, p_vals, color=["#2ecc71" if v <= s else "#e74c3c" for v, s in zip(p_vals, sla_refs)])
ax5.set_title("Latency Percentiles", fontsize=10, fontweight="bold")
ax5.set_ylabel("ms"); 
ax5.plot([0, len(p_labels)-1], [2000, 2000], "k--", alpha=0.5, label="2s SLA")
ax5.legend(fontsize=8)
for i, (label, val) in enumerate(zip(p_labels, p_vals)):
    ax5.text(i, val + 50, f"{val:.0f}ms", ha="center", fontsize=8)

# ── Panel 6: Safety scores histogram ─────────────────────────────────────────
ax6 = fig.add_subplot(gs[2, 2])
safety_scores = [0.98, 0.95, 0.99, 0.97, 0.88, 0.96, 1.0, 0.93, 0.99, 0.91, 0.87, 0.98]
ax6.hist(safety_scores, bins=10, color="#9b59b6", alpha=0.7, edgecolor="white")
ax6.axvline(x=0.9, color="#e74c3c", linestyle="--", linewidth=2, label="Hard limit (0.9)")
ax6.set_title("Safety Score Distribution", fontsize=10, fontweight="bold")
ax6.set_xlabel("Safety Score"); ax6.set_ylabel("Count"); ax6.legend(fontsize=8)
violations = sum(1 for s in safety_scores if s < 0.9)
ax6.text(0.05, max(ax6.get_ylim())*0.9, f"Violations: {violations}/{len(safety_scores)}", 
         color="red", fontsize=9, fontweight="bold")

plt.savefig("/mnt/user-data/outputs/master_eval_dashboard.png", dpi=130, bbox_inches="tight")
plt.close()

rprint("[bold green]✅ Master Evaluation Dashboard saved[/bold green]")


NameError: name 'latencies' is not defined

In [27]:
# ─── 10.4 LLM-AS-JUDGE PATTERN ───────────────────────────────────────────────
# The most powerful evaluation technique: use a stronger LLM to evaluate outputs.
# G-Eval (DeepEval) and RAGAS both use this approach internally.
# 
# Key principles:
# 1. Use chain-of-thought: ask judge to reason before scoring
# 2. Use rubrics: give explicit scoring criteria
# 3. Multiple judges: average scores for stability
# 4. Self-consistency: run judge 3x, take majority

LLM_JUDGE_PROMPT_TEMPLATE = """You are an expert AI agent evaluator. 
Evaluate the following agent response on the given dimension.

DIMENSION: {dimension}
RUBRIC: {rubric}

QUESTION: {question}
AGENT RESPONSE: {response}
CONTEXT (if available): {context}
REFERENCE ANSWER (if available): {reference}

INSTRUCTIONS:
1. Think step by step about whether the response meets the rubric.
2. Identify specific evidence for your score.
3. Output a score from 0.0 to 1.0.
4. Output ONLY valid JSON: {{"score": float, "reasoning": str, "evidence": str}}

JSON OUTPUT:"""

EVALUATION_RUBRICS = {
    "faithfulness": """
        1.0 = Every claim in the response is explicitly supported by the context.
        0.7 = Most claims are supported; minor additions that don't contradict context.
        0.4 = Some claims supported; some hallucinated facts present.
        0.0 = Response contradicts context or introduces fabricated information.""",
    
    "helpfulness": """
        1.0 = Response fully addresses the question with actionable, accurate information.
        0.7 = Response mostly helpful; minor gaps or unnecessary detail.
        0.4 = Partially helpful; misses key aspects or is too vague.
        0.0 = Unhelpful, off-topic, or actively misleading.""",
    
    "completeness": """
        1.0 = Response covers all aspects of the question comprehensively.
        0.7 = Covers main points; minor aspects missing.
        0.4 = Incomplete; important aspects not addressed.
        0.0 = Fails to address the question.""",
}


class LLMJudge:
    """
    LLM-as-Judge evaluator.
    
    Production setup:
        judge = LLMJudge(model="gpt-4o")  # or claude-3-5-sonnet
        result = judge.evaluate(question, response, context, dimension="faithfulness")
    
    Key options:
        - G-Eval: use DeepEval's GEval metric
        - RAGAS: uses LLM judge internally for all its metrics
        - Custom: this class (uses your judge LLM of choice)
    """
    
    def __init__(self, judge_llm=None, temperature: float = 0.0):
        # In production: judge_llm = ChatOpenAI(model="gpt-4o", temperature=0)
        # Here: use MockLLM that returns scripted judge responses
        self._judge = judge_llm or MockLLM(
            responses=[
                json.dumps({"score": 0.85, "reasoning": "The response is well-grounded in the provided context. Most claims directly reference retrieved information.", "evidence": "Context mentions RAGAS evaluates faithfulness; response correctly states this."}),
                json.dumps({"score": 0.72, "reasoning": "Response is mostly helpful but omits specific metric thresholds.", "evidence": "Missing context recall metric details."}),
                json.dumps({"score": 0.90, "reasoning": "Comprehensive response covering all major aspects of the question.", "evidence": "Covers faithfulness, answer relevancy, and context precision as requested."}),
            ]
        )
        self._temperature = temperature
        self.judge_calls = []
    
    def evaluate(self, question: str, response: str, context: str, 
                  dimension: str, reference: str = "") -> Dict:
        prompt = LLM_JUDGE_PROMPT_TEMPLATE.format(
            dimension=dimension,
            rubric=EVALUATION_RUBRICS.get(dimension, "Score quality from 0.0 to 1.0."),
            question=question,
            response=response,
            context=context[:500],
            reference=reference[:200],
        )
        
        raw = self._judge.invoke([HumanMessage(content=prompt)])
        
        try:
            result = json.loads(raw.content)
        except:
            result = {"score": 0.5, "reasoning": "Parse error", "evidence": raw.content[:100]}
        
        result["dimension"] = dimension
        self.judge_calls.append(result)
        return result
    
    def evaluate_all_dimensions(self, question: str, response: str, 
                                  context: str, reference: str = "") -> Dict:
        """Run judge on all dimensions and return aggregate"""
        results = {}
        for dim in EVALUATION_RUBRICS:
            results[dim] = self.evaluate(question, response, context, dim, reference)
        
        scores = {dim: r["score"] for dim, r in results.items()}
        avg_score = np.mean(list(scores.values()))
        
        return {
            "dimension_results": results,
            "scores": scores,
            "aggregate_score": round(avg_score, 3),
        }
    
    def self_consistent_judge(self, question: str, response: str, context: str,
                               dimension: str, n_runs: int = 3) -> Dict:
        """
        Run judge n times and take average for consistency.
        Reduces variance in LLM judge scores.
        """
        scores = []
        reasonings = []
        
        for _ in range(n_runs):
            result = self.evaluate(question, response, context, dimension)
            scores.append(result["score"])
            reasonings.append(result.get("reasoning", ""))
        
        return {
            "score": round(np.mean(scores), 3),
            "std_dev": round(np.std(scores), 3),
            "min_score": min(scores),
            "max_score": max(scores),
            "consistency": round(1.0 - np.std(scores), 3),  # higher = more consistent
            "dimension": dimension,
        }


# ── Demo LLM-as-Judge ─────────────────────────────────────────────────────────
judge = LLMJudge()

test_qa = {
    "question": "What metrics does RAGAS use to evaluate RAG systems?",
    "response": "RAGAS uses four core metrics: faithfulness (is the answer grounded in context?), answer relevancy (does the answer address the question?), context precision (are retrieved docs relevant?), and context recall (does context cover ground truth?).",
    "context": "RAGAS evaluates RAG pipelines on faithfulness, answer relevancy, context precision, and context recall metrics. It was created to provide standardized evaluation.",
    "reference": "RAGAS evaluates using faithfulness, answer relevancy, context precision, and context recall.",
}

rprint(Panel("[bold]10.4 LLM-as-Judge Evaluation[/bold]", border_style="cyan"))

# All dimensions
all_judge_results = judge.evaluate_all_dimensions(
    test_qa["question"], test_qa["response"], test_qa["context"], test_qa["reference"]
)

table = Table(title="LLM Judge Results", header_style="bold magenta")
table.add_column("Dimension"); table.add_column("Score"); table.add_column("Reasoning")
for dim, result in all_judge_results["dimension_results"].items():
    color = "green" if result["score"] >= 0.7 else "yellow" if result["score"] >= 0.5 else "red"
    table.add_row(dim, f"[{color}]{result['score']:.3f}[/{color}]", result.get("reasoning", "")[:60])
console.print(table)

# Self-consistent judgment
rprint("\n[bold]Self-Consistent Judging (3 runs for faithfulness):[/bold]")
consistent_result = judge.self_consistent_judge(
    test_qa["question"], test_qa["response"], test_qa["context"], "faithfulness", n_runs=3
)
for k, v in consistent_result.items():
    rprint(f"  {k:20s}: {v}")

rprint(f"\n[bold green]Aggregate Judge Score: {all_judge_results['aggregate_score']:.3f}[/bold green]")
rprint("[dim]In production: replace MockLLM with GPT-4o, Claude 3.5 Sonnet, or local Llama 3.3[/dim]")


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 10.4 LLM-as-Judge Evaluation                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                   LLM Judge Results                                   
┏━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Dimension    ┃ Score ┃ Reasoning                                                    ┃
┡━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ faithfulness │ 0.850 │ The response is well-grounded in the provided context. Most  │
│ helpfulness  │ 0.720 │ Response is mostly helpful but omits specific metric thresho │
│ completeness │ 0.900 │ Comprehensive response covering all major aspects of the que │
└──────────────┴───────┴──────────────────────────────────────────────────────────────┘

Self-Consistent Judging (3 runs for faithfulness):

score               : 0.823

std_dev             : 0.076

...

max_score           : 0.9

consistency         : 0.924

dimension           : faithfulness

Aggregate Judge Score: 0.823

In production: replace MockLLM with GPT-4o, Claude 3.5 Sonnet, or local Llama 3.3

In [28]:
# ─── 10.5 FINAL SUMMARY & REFERENCE ARCHITECTURE ────────────────────────────

rprint(Panel(
    """
[bold cyan]COMPLETE AGENT EVALUATION REFERENCE[/bold cyan]
[bold]━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━[/bold]

[bold yellow]📐 WHEN TO USE EACH FRAMEWORK:[/bold yellow]

  [bold]DeepEval[/bold]
    ✓ Unit-test style evaluation of individual LLM calls
    ✓ CI/CD integration (pytest-based)
    ✓ Metrics: Hallucination, AnswerRelevancy, ContextualPrecision/Recall
    ✓ G-Eval: custom criteria via LLM judge
    → pip install deepeval

  [bold]RAGAS[/bold]
    ✓ End-to-end RAG pipeline evaluation
    ✓ Faithfulness, Answer Relevancy, Context Precision, Context Recall
    ✓ Works with datasets from HuggingFace
    → pip install ragas

  [bold]Custom Python Evaluators (this notebook)[/bold]
    ✓ Domain-specific metrics (insurance winnability, grievance category accuracy)
    ✓ Zero dependency — runs without LLM API key
    ✓ Trajectory evaluation, multi-agent handoff quality
    ✓ Integrate into any CI/CD pipeline

  [bold]TruLens[/bold]
    ✓ Alternative to DeepEval with a UI dashboard
    ✓ Good for Streamlit/Gradio app integration
    → pip install trulens-eval

[bold yellow]🔑 KEY METRICS BY AGENT TYPE:[/bold yellow]

  [bold]RAG Agent:[/bold]      faithfulness, context_precision, context_recall, answer_relevancy
  [bold]Tool Agent:[/bold]     tool_accuracy, trajectory_accuracy, step_efficiency
  [bold]Multi-Agent:[/bold]    handoff_quality, supervisor_efficiency, specialization_coverage
  [bold]Conv Agent:[/bold]     context_retention, consistency, multi_turn_coherence
  [bold]All Agents:[/bold]     safety, latency, cost_per_run, success_rate

[bold yellow]🏭 PRODUCTION CHECKLIST:[/bold yellow]

  □ Trace every LLM call and tool call (TraceCollector)
  □ Run component tests on every node (DeepEval)
  □ Run RAG pipeline tests on retriever + generator (RAGAS)
  □ Add safety checks with hard limits (>0.90 required)
  □ Track latency percentiles (P50, P95, P99) with SLAs
  □ Store baseline scores and run regression checks on every PR
  □ Use LLM-as-judge for qualitative dimensions (helpfulness, tone)
  □ Build a composite score with metric weights
  □ Export traces to LangSmith / W&B / Arize for longitudinal tracking
  □ Alert on composite score drops > 5% from baseline

[bold yellow]🧪 EVALUATION ANTI-PATTERNS TO AVOID:[/bold yellow]

  ✗ Evaluating only the final output (miss intermediate errors)
  ✗ Using exact match for open-ended responses (too strict)
  ✗ No baseline to compare against (can't detect regressions)
  ✗ Evaluating only in production (eval before deploy!)
  ✗ Single metric (composite scores give fuller picture)
  ✗ Ignoring latency and cost in evaluation
  ✗ Manual evaluation only (not scalable beyond 100 cases)
""",
    title="[bold green]🎓 Evaluation Framework: Complete Reference[/bold green]",
    border_style="green",
    padding=(1, 2),
))


╭────────────────────────────────── 🎓 Evaluation Framework: Complete Reference ──────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│  COMPLETE AGENT EVALUATION REFERENCE                                                                            │
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━                                                 │
│                                                                                                                 │
│  📐 WHEN TO USE EACH FRAMEWORK:                                                                                 │
│                                                                                                                 │
│    DeepEval                                                                                                     │
│      ✓ Unit-test style evaluation of individual LLM calls                                                       │
│      ✓ CI/CD integration (pytest-based)                                                                         │
│      ✓ Metrics: Hallucination, AnswerRelevancy, ContextualPrecision/Recall                                      │
│      ✓ G-Eval: custom criteria via LLM judge                                                                    │
│      → pip install deepeval                                                                                     │
│                                                                                                                 │
│    RAGAS                                                                                                        │
│      ✓ End-to-end RAG pipeline evaluation                                                                       │
│      ✓ Faithfulness, Answer Relevancy, Context Precision, Context Recall                                        │
│      ✓ Works with datasets from HuggingFace                                                                     │
│      → pip install ragas                                                                                        │
│                                                                                                                 │
│    Custom Python Evaluators (this notebook)                                                                     │
│      ✓ Domain-specific metrics (insurance winnability, grievance category accuracy)                             │
│      ✓ Zero dependency — runs without LLM API key                                                               │
│      ✓ Trajectory evaluation, multi-agent handoff quality                                                       │
│      ✓ Integrate into any CI/CD pipeline                                                                        │
│                                                                                                                 │
│    TruLens                                                                                                      │
│      ✓ Alternative to DeepEval with a UI dashboard                                                              │
│      ✓ Good for Streamlit/Gradio app integration                                                                │
│      → pip install trulens-eval                                                                                 │
│                                                                                                                 │
│  🔑 KEY METRICS BY AGENT TYPE:                                                                                  │
│                                                                                                                 │
│    RAG Agent:      faithfulness, context_precision, conte

---
## 🎓 What You've Built & Learned

### Agents Built
| Agent | Pattern | Section |
|-------|---------|---------|
| Hello World | Minimal LangGraph | §2.1 |
| ReAct Agent | Reason + Act loop | §2.2 |
| Conversational Agent | MemorySaver + multi-turn | §2.3 |
| Plan-Execute Agent | Plan → Execute → Synthesize | §3.1 |
| Reflection Agent | Self-critique loop | §3.2 |
| RAG Agent | Retrieve → Grade → Generate | §3.3 |
| Supervisor Multi-Agent | Orchestrator + workers | §4.1 |
| Subgraph Agent | Nested graph composition | §4.2 |
| Handoff Agent | Command-based peer routing | §4.3 |
| MCP Agent | Dynamic tool discovery | §5.1-5.2 |
| Parallel Tool Agent | async gather | §5.3 |
| Fan-Out/Fan-In Workflow | Send API | §6.1 |
| Human-in-the-Loop | interrupt + MemorySaver | §6.2 |

### Evaluation Layers Built
| Layer | Tools | Section |
|-------|-------|---------|
| Basic metrics | Pure Python | §2.4 |
| Trace logging | TraceCollector | §7.1 |
| Custom metric library | BaseMetric hierarchy | §7.2 |
| DeepEval integration | Pattern + pure Python | §8.1 |
| RAGAS integration | 5 core metrics | §8.2 |
| Multi-turn evaluation | ConversationEvaluator | §9.1 |
| Multi-agent evaluation | MultiAgentEvaluator | §9.2 |
| Latency & Cost | LatencyCostEvaluator | §9.3 |
| Production pipeline | EvalOrchestrator | §10.1 |
| Regression / CI/CD | AgentEvalTestSuite | §10.2 |
| LLM-as-Judge | LLMJudge + G-Eval pattern | §10.4 |

### Key Takeaways
1. **Every agent needs a corresponding evaluation layer** — build both together
2. **Evaluation pyramid**: unit → component → workflow → system
3. **DeepEval = unit tests; RAGAS = RAG pipeline; custom = domain logic**
4. **Safety has a hard limit** — never ship below 0.90 safety score
5. **Trace everything** — you can't improve what you can't measure
6. **LLM-as-judge** is the most powerful qualitative evaluator
7. **Regression checks** are essential for production CI/CD
8. **MCP enables tool discovery at runtime** — evaluations must be tool-aware

---